<a href="https://colab.research.google.com/github/cduplan59/CFT_analysis/blob/main/cell4_spectral_dimension_robustness_colab_nreal50_TEST1K_DRIVE_RESUME_V2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/cduplan59/CFT_analysis/blob/main/cell4_spectral_dimension_robustness_colab_nreal50_TEST1K.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ROSG — Test 1K numérique
## Spectral cell-4 crossover with Kigami-type resistance conductances + Hill comparator

This notebook rewrites Test 1 to consolidate the chain:

\[
K_n^{\mathrm{cell4}}
\rightarrow
\mathcal{E}_Z(f,f)
\rightarrow
L_Z
\rightarrow
P(u;Z)
\rightarrow
d_s^{\mathrm{eff}}(Z).
\]

The main improvement is to replace the purely phenomenological vertical weights with an option of **effective resistance conductances**:

\[
\mathcal{E}_Z(f,f)=\sum_{i\sim j} c_{ij}^{(n)}(Z)[f_i-f_j]^2,
\qquad
c_{ij}^{(n)}(Z)\sim \frac{1}{R_{ij}^{(n)}(Z)}.
\]

The term \(R_{ij}^{(n)}\) is implemented here as a **local renormalized resistance proxy** associated with the cell-4 hierarchy. This is not a complete analytical proof of Kigami; it is an operational extension of Test 1.

The notebook also adds a **saturating power / Hill** comparator:

\[
d_s(Z)=d_{\rm UV}+(d_{\rm IR}-d_{\rm UV})
\frac{e^{pZ}}{e^{pZ}+e^{pZ_c}},
\]

which is equivalent to a logistic function in the variable \(Z=\ln(\Delta\tau/t_P)\).


In [ ]:
# ============================================================
# 0. Setup Colab
# ============================================================
# !pip -q install numpy scipy pandas matplotlib

import os, math, time, json, warnings, shutil, glob, zipfile
from dataclasses import dataclass, asdict, replace
from typing import Tuple, Dict, Optional, List, Any, Set
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import scipy.sparse as sp
from scipy.sparse.linalg import eigsh, expm_multiply
from scipy.optimize import curve_fit
from scipy.signal import savgol_filter

np.set_printoptions(precision=5, suppress=True)
warnings.filterwarnings("ignore", category=RuntimeWarning)
print("OK imports")


OK imports


In [ ]:
# ============================================================
# 1. Configuration globale + persistance Drive
# ============================================================

@dataclass
class Cell4Config:
    n_subdiv: int = 4
    layers: int = 8
    overlay_mode: str = "multiplex"       # "plane2d" ou "multiplex"
    torus_xy: bool = True
    torus_layers: bool = False

    z_min: float = -1.0
    z_max: float = 5.0
    n_z: int = 17

    theta_center: float = 2.0
    theta_width_hier: float = 1.1
    jitter: float = 0.35

    w_plane: float = 1.0
    w_vertical_lo: float = 0.0
    w_vertical_hi: float = 1.0

    # "phenomenological" = Test 1 initial ; "kigami_local" = c_ij ~ 1/R_ij proxy.
    conductance_rule: str = "kigami_local"

    resistance_scale: float = 1.0
    resistance_hier_power: float = 1.0
    resistance_eps: float = 1e-12
    normalize_active_vertical_mean: bool = True

    heat_method: str = "hutchinson"       # "exact", "eigs", "hutchinson", "auto"
    n_probes: int = 24
    u_min: float = 0.08
    u_max: float = 80.0
    n_u: int = 34
    full_dense_limit: int = 700

    n_realizations: int = 50
    random_seed: int = 12345

    p_high: float = 0.85
    p_low_floor: float = 0.015
    min_window_points: int = 6

    smooth_ds: bool = True
    savgol_window: int = 7
    savgol_poly: int = 2

    # Dossier local Colab : rapide mais volatil.
    out_dir: str = "/content/cell4_test1K_kigami_outputs"

    # Dossier Drive : persistant. Les résultats y sont copiés après chaque variante
    # et pendant les scans longs via *_PARTIAL_raw.csv.
    use_google_drive: bool = True
    drive_mount_point: str = "/content/drive"
    drive_out_dir: str = "/content/drive/MyDrive/DFGQ_Test1K/cell4_test1K_kigami_outputs"

    # Relance/reprise.
    resume_from_drive: bool = True
    force_rerun_completed: bool = False
    checkpoint_after_each_Z: bool = True
    mirror_after_each_run: bool = True
    zip_after_each_run: bool = True

CFG = Cell4Config()
print(json.dumps(asdict(CFG), indent=2))


{
  "n_subdiv": 4,
  "layers": 8,
  "overlay_mode": "multiplex",
  "torus_xy": true,
  "torus_layers": false,
  "z_min": -1.0,
  "z_max": 5.0,
  "n_z": 17,
  "theta_center": 2.0,
  "theta_width_hier": 1.1,
  "jitter": 0.35,
  "w_plane": 1.0,
  "w_vertical_lo": 0.0,
  "w_vertical_hi": 1.0,
  "conductance_rule": "kigami_local",
  "resistance_scale": 1.0,
  "resistance_hier_power": 1.0,
  "resistance_eps": 1e-12,
  "normalize_active_vertical_mean": true,
  "heat_method": "hutchinson",
  "n_probes": 24,
  "u_min": 0.08,
  "u_max": 80.0,
  "n_u": 34,
  "full_dense_limit": 700,
  "n_realizations": 50,
  "random_seed": 12345,
  "p_high": 0.85,
  "p_low_floor": 0.015,
  "min_window_points": 6,
  "smooth_ds": true,
  "savgol_window": 7,
  "savgol_poly": 2,
  "out_dir": "/content/cell4_test1K_kigami_outputs",
  "use_google_drive": true,
  "drive_mount_point": "/content/drive",
  "drive_out_dir": "/content/drive/MyDrive/DFGQ_Test1K/cell4_test1K_kigami_outputs",
  "resume_from_drive": true,
  "for

In [ ]:
# ============================================================
# 2. Flags d'exécution
# ============================================================

RUN_AUTOTESTS = True
RUN_MAIN = True
RUN_NEGATIVE_CONTROL = True
RUN_PHENOM_COMPARISON = True

# Avec n_realizations=50, le calcul est long.
# Le notebook sauvegarde maintenant chaque variante sur Drive, puis reprend sans recalculer.
RUN_FAST_ROBUSTNESS = True
RUN_PUBLICATION_SWEEP = True
PLOT_EACH_VARIANT = False

print({
    "RUN_AUTOTESTS": RUN_AUTOTESTS,
    "RUN_MAIN": RUN_MAIN,
    "RUN_NEGATIVE_CONTROL": RUN_NEGATIVE_CONTROL,
    "RUN_PHENOM_COMPARISON": RUN_PHENOM_COMPARISON,
    "RUN_FAST_ROBUSTNESS": RUN_FAST_ROBUSTNESS,
    "RUN_PUBLICATION_SWEEP": RUN_PUBLICATION_SWEEP,
    "PLOT_EACH_VARIANT": PLOT_EACH_VARIANT,
    "resume_from_drive": CFG.resume_from_drive,
    "drive_out_dir": CFG.drive_out_dir,
})


{'RUN_AUTOTESTS': True, 'RUN_MAIN': True, 'RUN_NEGATIVE_CONTROL': True, 'RUN_PHENOM_COMPARISON': True, 'RUN_FAST_ROBUSTNESS': True, 'RUN_PUBLICATION_SWEEP': True, 'PLOT_EACH_VARIANT': False, 'resume_from_drive': True, 'drive_out_dir': '/content/drive/MyDrive/DFGQ_Test1K/cell4_test1K_kigami_outputs'}


In [ ]:
# ============================================================
# 3. Utilitaires mathématiques et indexation
# ============================================================

def L_from_cfg(cfg: Cell4Config) -> int:
    return 2 ** int(cfg.n_subdiv)

def node_id(x: int, y: int, layer: int, L: int, layers: int) -> int:
    return (layer * L + y) * L + x

def trailing_zeros_power2_index(a: int, n_subdiv: int) -> int:
    v = int(a) + 1
    c = 0
    while v % 2 == 0 and c < n_subdiv:
        c += 1
        v //= 2
    return c

def local_hierarchy_level(x: int, y: int, cfg: Cell4Config) -> int:
    return int(max(trailing_zeros_power2_index(x, cfg.n_subdiv), trailing_zeros_power2_index(y, cfg.n_subdiv)))

def edge_threshold_Z(x: int, y: int, layer: int, cfg: Cell4Config, rng: np.random.Generator) -> float:
    h = local_hierarchy_level(x, y, cfg)
    h_centered = (cfg.n_subdiv / 2.0 - h) / max(cfg.n_subdiv, 1)
    theta = cfg.theta_center + cfg.theta_width_hier * h_centered
    theta += cfg.jitter * rng.normal()
    return float(theta)

def hard_activation(Z: float, theta: float) -> float:
    return 1.0 if Z >= theta else 0.0

def local_kigami_resistance_proxy(x: int, y: int, layer: int, cfg: Cell4Config) -> float:
    """
    Proxy local de résistance effective de type Kigami.
    R(h)=resistance_scale/(1+h)^resistance_hier_power.
    Les jonctions plus structurées ont une résistance plus faible, donc une conductance plus forte.
    """
    h = local_hierarchy_level(x, y, cfg)
    R = cfg.resistance_scale / ((1.0 + h) ** cfg.resistance_hier_power)
    return float(max(R, cfg.resistance_eps))

def vertical_raw_conductance(x: int, y: int, layer: int, Z: float, theta: float, cfg: Cell4Config) -> float:
    a = hard_activation(Z, theta)
    if a <= 0:
        return float(cfg.w_vertical_lo)
    if cfg.conductance_rule == "phenomenological":
        return float(cfg.w_vertical_hi)
    if cfg.conductance_rule == "kigami_local":
        R = local_kigami_resistance_proxy(x, y, layer, cfg)
        return float(1.0 / (R + cfg.resistance_eps))
    raise ValueError(f"Unknown conductance_rule={cfg.conductance_rule}")

def tanh_profile(Z, d_uv, d_ir, z_th, width):
    Z = np.asarray(Z, dtype=float)
    width = np.maximum(width, 1e-9)
    return d_uv + 0.5 * (d_ir - d_uv) * (1.0 + np.tanh((Z - z_th) / width))

def logistic_profile(Z, d_uv, d_ir, z_th, width):
    Z = np.asarray(Z, dtype=float)
    width = np.maximum(width, 1e-9)
    return d_uv + (d_ir - d_uv) / (1.0 + np.exp(-(Z - z_th) / width))

def hill_saturating_profile(Z, d_uv, d_ir, z_c, p):
    Z = np.asarray(Z, dtype=float)
    p = np.maximum(p, 1e-9)
    return d_uv + (d_ir - d_uv) / (1.0 + np.exp(-p * (Z - z_c)))

def pure_power_shifted_profile(Z, a, b, p):
    Z = np.asarray(Z, dtype=float)
    X = Z - np.min(Z) + 1e-6
    return a + b * np.power(X, p)

def constant_profile(Z, c):
    return np.zeros_like(np.asarray(Z, dtype=float)) + c

def linear_profile(Z, a, b):
    return a + b * np.asarray(Z, dtype=float)


In [ ]:
# ============================================================
# 4. Construction cell-4 : graphe pondéré et Laplacien
# ============================================================

def build_cell4_adjacency(cfg: Cell4Config, Z: float, seed: Optional[int] = None) -> sp.csr_matrix:
    L = L_from_cfg(cfg)
    layers = 1 if cfg.overlay_mode == "plane2d" else int(cfg.layers)
    N = L * L * layers
    rng = np.random.default_rng(cfg.random_seed if seed is None else seed)
    rows, cols, data = [], [], []
    vertical_edges = []

    def add_undirected(i, j, w):
        if i == j or w == 0:
            return
        rows.extend([i, j]); cols.extend([j, i]); data.extend([float(w), float(w)])

    for ell in range(layers):
        for y in range(L):
            for x in range(L):
                i = node_id(x, y, ell, L, layers)
                if x + 1 < L:
                    add_undirected(i, node_id(x + 1, y, ell, L, layers), cfg.w_plane)
                elif cfg.torus_xy and L > 2:
                    add_undirected(i, node_id(0, y, ell, L, layers), cfg.w_plane)
                if y + 1 < L:
                    add_undirected(i, node_id(x, y + 1, ell, L, layers), cfg.w_plane)
                elif cfg.torus_xy and L > 2:
                    add_undirected(i, node_id(x, 0, ell, L, layers), cfg.w_plane)

    if cfg.overlay_mode == "multiplex" and layers > 1 and cfg.w_vertical_hi != 0:
        for ell in range(layers - 1):
            for y in range(L):
                for x in range(L):
                    theta = edge_threshold_Z(x, y, ell, cfg, rng)
                    w_raw = vertical_raw_conductance(x, y, ell, Z, theta, cfg)
                    if w_raw != 0:
                        vertical_edges.append((node_id(x, y, ell, L, layers), node_id(x, y, ell + 1, L, layers), w_raw))
        if cfg.torus_layers and layers > 2:
            ell = layers - 1
            for y in range(L):
                for x in range(L):
                    theta = edge_threshold_Z(x, y, ell, cfg, rng)
                    w_raw = vertical_raw_conductance(x, y, ell, Z, theta, cfg)
                    if w_raw != 0:
                        vertical_edges.append((node_id(x, y, ell, L, layers), node_id(x, y, 0, L, layers), w_raw))

    if vertical_edges:
        weights = np.asarray([e[2] for e in vertical_edges], dtype=float)
        if cfg.conductance_rule == "kigami_local" and cfg.normalize_active_vertical_mean:
            scale = max(cfg.w_vertical_hi, 1e-12) / max(float(weights.mean()), 1e-12)
        else:
            scale = 1.0
        for i, j, w in vertical_edges:
            add_undirected(i, j, scale * w)

    W = sp.coo_matrix((data, (rows, cols)), shape=(N, N), dtype=float).tocsr()
    W.sum_duplicates()
    return W

def laplacian_from_adjacency(W: sp.csr_matrix) -> sp.csr_matrix:
    deg = np.asarray(W.sum(axis=1)).ravel()
    return sp.diags(deg, 0, format="csr") - W

def graph_stats(W: sp.csr_matrix) -> Dict[str, float]:
    from scipy.sparse.csgraph import connected_components
    n_components, labels = connected_components(W, directed=False)
    counts = np.bincount(labels)
    return {
        "N": int(W.shape[0]),
        "E_undirected": int(W.nnz // 2),
        "avg_degree_weighted": float(np.asarray(W.sum(axis=1)).mean()),
        "n_components": int(n_components),
        "lcc_ratio": float(counts.max() / W.shape[0]),
    }

def count_vertical_edges(W: sp.csr_matrix, cfg: Cell4Config) -> int:
    L = L_from_cfg(cfg)
    layers = 1 if cfg.overlay_mode == "plane2d" else int(cfg.layers)
    if layers <= 1: return 0
    step = L * L
    coo = W.tocoo()
    return int(np.sum(np.abs(coo.row - coo.col) == step) // 2)


In [ ]:
# ============================================================
# 5. Trace de chaleur et extraction de d_s
# ============================================================

def heat_trace_exact(Lap: sp.csr_matrix, u_grid: np.ndarray) -> Tuple[np.ndarray, Dict[str, Any]]:
    evals = np.linalg.eigvalsh(Lap.toarray())
    evals = np.maximum(evals, 0.0)
    P = np.exp(-np.outer(u_grid, evals)).mean(axis=1)
    return P, {"method": "exact", "n_eigs": int(len(evals))}

def heat_trace_eigs(Lap: sp.csr_matrix, u_grid: np.ndarray, k: int = 400) -> Tuple[np.ndarray, Dict[str, Any]]:
    N = Lap.shape[0]
    k = int(min(max(10, k), N - 2))
    evals = eigsh(Lap, k=k, which="SM", return_eigenvectors=False, tol=1e-6)
    evals = np.sort(np.maximum(evals, 0.0))
    P = np.exp(-np.outer(u_grid, evals)).sum(axis=1) / N
    return P, {"method": "eigs", "n_eigs": int(len(evals))}

def heat_trace_hutchinson(Lap: sp.csr_matrix, u_grid: np.ndarray, n_probes: int = 24, seed: int = 0) -> Tuple[np.ndarray, Dict[str, Any]]:
    N = Lap.shape[0]
    rng = np.random.default_rng(seed)
    B = rng.choice([-1.0, 1.0], size=(N, int(n_probes)))
    P = []
    for u in u_grid:
        Y = expm_multiply((-float(u)) * Lap, B)
        tr_est = np.sum(B * Y) / B.shape[1]
        P.append(tr_est / N)
    return np.clip(np.asarray(P, dtype=float), 1e-300, np.inf), {"method": "hutchinson", "n_probes": int(n_probes)}

def compute_heat_trace(cfg: Cell4Config, Lap: sp.csr_matrix, u_grid: np.ndarray, seed: int = 0):
    N = Lap.shape[0]
    if cfg.heat_method == "exact" or (cfg.heat_method == "auto" and N <= cfg.full_dense_limit):
        if N <= cfg.full_dense_limit:
            return heat_trace_exact(Lap, u_grid)
        return heat_trace_hutchinson(Lap, u_grid, cfg.n_probes, seed)
    if cfg.heat_method == "eigs":
        return heat_trace_eigs(Lap, u_grid, k=min(500, max(30, N // 3)))
    if cfg.heat_method == "hutchinson":
        return heat_trace_hutchinson(Lap, u_grid, cfg.n_probes, seed)
    raise ValueError(f"Unknown heat_method={cfg.heat_method}")

def ds_from_heat_trace(u_grid: np.ndarray, P: np.ndarray, cfg: Cell4Config) -> np.ndarray:
    logu = np.log(u_grid)
    logP = np.log(np.clip(P, 1e-300, np.inf))
    if cfg.smooth_ds and len(logP) >= cfg.savgol_window:
        win = cfg.savgol_window + (cfg.savgol_window % 2 == 0)
        win = min(win, len(logP) if len(logP) % 2 == 1 else len(logP) - 1)
        if win >= cfg.savgol_poly + 2:
            logP = savgol_filter(logP, window_length=win, polyorder=cfg.savgol_poly)
    return -2.0 * np.gradient(logP, logu)

def effective_ds_from_curve(u_grid: np.ndarray, P: np.ndarray, ds_curve: np.ndarray, cfg: Cell4Config, N: int) -> Dict[str, Any]:
    p_low = max(cfg.p_low_floor, 5.0 / max(N, 1))
    mask = (P < cfg.p_high) & (P > p_low) & np.isfinite(ds_curve) & (ds_curve > -1.0) & (ds_curve < 8.0)
    if mask.sum() < cfg.min_window_points:
        lo, hi = np.quantile(np.arange(len(u_grid)), [0.25, 0.75]).astype(int)
        mask = np.zeros_like(u_grid, dtype=bool); mask[lo:hi+1] = True
    vals = ds_curve[mask]
    return {
        "ds_eff_median": float(np.median(vals)),
        "ds_eff_mean": float(np.mean(vals)),
        "ds_eff_std": float(np.std(vals)),
        "window_points": int(mask.sum()),
        "u_window_min": float(u_grid[mask].min()),
        "u_window_max": float(u_grid[mask].max()),
        "mask": mask,
    }


In [ ]:
# ============================================================
# 6. Scan Z -> d_s^eff(Z), avec checkpoints partiels Drive
# ============================================================

def run_single_Z(cfg: Cell4Config, Z: float, realization: int = 0) -> Dict[str, Any]:
    seed = cfg.random_seed + 100_000 * realization + int(round((Z + 1000) * 1000))
    W = build_cell4_adjacency(cfg, Z=Z, seed=seed)
    Lap = laplacian_from_adjacency(W)
    u_grid = np.logspace(np.log10(cfg.u_min), np.log10(cfg.u_max), cfg.n_u)
    P, meta = compute_heat_trace(cfg, Lap, u_grid, seed=seed + 17)
    ds_curve = ds_from_heat_trace(u_grid, P, cfg)
    eff = effective_ds_from_curve(u_grid, P, ds_curve, cfg, N=W.shape[0])
    stats = graph_stats(W)
    stats["E_vertical"] = count_vertical_edges(W, cfg)
    out = {"Z": float(Z), "realization": int(realization), **stats, **{k: v for k, v in eff.items() if k != "mask"}, **meta,
           "u_grid": u_grid, "P": P, "ds_curve": ds_curve, "mask": eff["mask"]}
    return out

def _round_key_Z(z: float) -> float:
    return float(np.round(float(z), 12))

def _partial_raw_paths(tag: str, cfg: Cell4Config) -> Tuple[str, str]:
    return (
        os.path.join(cfg.out_dir, f"{tag}_PARTIAL_raw.csv"),
        os.path.join(cfg.drive_out_dir, f"{tag}_PARTIAL_raw.csv"),
    )

def _load_partial_raw(tag: str, cfg: Cell4Config) -> pd.DataFrame:
    local_partial, drive_partial = _partial_raw_paths(tag, cfg)
    candidates = [local_partial]
    if cfg.resume_from_drive:
        candidates.append(drive_partial)
    for p in candidates:
        if p and os.path.exists(p) and os.path.getsize(p) > 0:
            try:
                df = pd.read_csv(p)
                if len(df):
                    print(f"[resume] loaded partial raw for {tag}: {p} ({len(df)} rows)")
                    return df
            except Exception as exc:
                print(f"[resume-warning] could not read partial raw {p}: {exc}")
    return pd.DataFrame()

def _save_partial_raw(tag: str, cfg: Cell4Config, rows: List[Dict[str, Any]]):
    if not rows:
        return
    df = pd.DataFrame(rows)
    ensure_out_dir(cfg.out_dir)
    local_partial, drive_partial = _partial_raw_paths(tag, cfg)
    df.to_csv(local_partial, index=False)
    if cfg.use_google_drive and cfg.mirror_after_each_run:
        ensure_out_dir(cfg.drive_out_dir)
        try:
            shutil.copy2(local_partial, drive_partial)
        except Exception as exc:
            print(f"[drive-warning] partial mirror failed for {tag}: {exc}")

def run_scan(cfg: Cell4Config, verbose: bool = True, checkpoint_tag: Optional[str] = None) -> Tuple[pd.DataFrame, Dict[Tuple[float, int], Dict[str, Any]]]:
    """
    Scan robuste avec reprise :
    - si checkpoint_tag est fourni, le notebook lit *_PARTIAL_raw.csv local/Drive ;
    - il saute les couples (Z, realization) déjà calculés ;
    - il réécrit le CSV partiel après chaque valeur de Z.
    """
    z_grid = np.linspace(cfg.z_min, cfg.z_max, cfg.n_z)
    rows: List[Dict[str, Any]] = []
    curves: Dict[Tuple[float, int], Dict[str, Any]] = {}

    if checkpoint_tag:
        partial = _load_partial_raw(checkpoint_tag, cfg)
        if len(partial):
            rows.extend(partial.to_dict(orient="records"))

    done: Set[Tuple[float, int]] = set()
    for row in rows:
        if "Z" in row and "realization" in row:
            done.add((_round_key_Z(row["Z"]), int(row["realization"])))

    t0 = time.time()
    for iz, Z in enumerate(z_grid):
        new_rows_this_Z = 0
        for r in range(cfg.n_realizations):
            key = (_round_key_Z(Z), int(r))
            if key in done:
                continue
            out = run_single_Z(cfg, Z, realization=r)
            curves[(float(Z), r)] = {"u_grid": out.pop("u_grid"), "P": out.pop("P"), "ds_curve": out.pop("ds_curve"), "mask": out.pop("mask")}
            rows.append(out)
            done.add(key)
            new_rows_this_Z += 1

        if checkpoint_tag and cfg.checkpoint_after_each_Z and (new_rows_this_Z > 0 or iz == len(z_grid) - 1):
            _save_partial_raw(checkpoint_tag, cfg, rows)

        if verbose and ((iz + 1) % max(1, cfg.n_z // 5) == 0 or iz == cfg.n_z - 1):
            n_expected = cfg.n_z * cfg.n_realizations
            print(f"[scan] {iz+1:02d}/{cfg.n_z} Z values done | rows={len(rows)}/{n_expected} | elapsed={time.time()-t0:.1f}s")

    raw = pd.DataFrame(rows)
    if len(raw):
        raw = raw.drop_duplicates(subset=["Z", "realization"], keep="last").sort_values(["Z", "realization"]).reset_index(drop=True)

    expected = cfg.n_z * cfg.n_realizations
    if len(raw) != expected:
        print(f"[warning] incomplete scan for {checkpoint_tag}: {len(raw)}/{expected} rows. Aggregation may be partial.")
    return raw, curves

def aggregate_scan(raw: pd.DataFrame) -> pd.DataFrame:
    g = raw.groupby("Z", as_index=False)
    agg = g.agg(
        ds_mean=("ds_eff_mean", "mean"),
        ds_median=("ds_eff_median", "mean"),
        ds_std=("ds_eff_mean", "std"),
        ds_sem=("ds_eff_mean", lambda x: np.std(x, ddof=1) / math.sqrt(len(x)) if len(x) > 1 else 0.0),
        lcc_ratio=("lcc_ratio", "mean"),
        n_components=("n_components", "mean"),
        E_undirected=("E_undirected", "mean"),
        E_vertical=("E_vertical", "mean"),
        window_points=("window_points", "mean"),
        n_rows=("realization", "count"),
    )
    return agg.fillna(0.0)


In [ ]:
# ============================================================
# 7. Ajustement des modèles : constant, linéaire, puissance, tanh, logistique, Hill
# ============================================================

def aic_bic(y: np.ndarray, yhat: np.ndarray, k: int) -> Tuple[float, float, float]:
    n = len(y)
    rss = float(np.sum((np.asarray(y) - np.asarray(yhat)) ** 2))
    rss_safe = max(rss, 1e-300)
    return rss, float(n * np.log(rss_safe / n) + 2 * k), float(n * np.log(rss_safe / n) + k * np.log(n))

def _safe_curve_fit(func, Z, y, p0, bounds=(-np.inf, np.inf), maxfev=100000):
    try:
        popt, pcov = curve_fit(func, Z, y, p0=p0, bounds=bounds, maxfev=maxfev)
        return popt, pcov, None
    except Exception as e:
        return None, None, str(e)

def fit_profile_models(df: pd.DataFrame, y_col: str = "ds_mean") -> Tuple[pd.DataFrame, Dict[str, np.ndarray]]:
    Z = df["Z"].to_numpy(dtype=float)
    y = df[y_col].to_numpy(dtype=float)
    ymin, ymax = float(np.min(y)), float(np.max(y))
    zmid = float(Z[np.argmin(np.abs(y - 0.5 * (ymin + ymax)))])
    width0 = max((Z.max() - Z.min()) / 6.0, 0.1)
    models, preds = [], {}

    def add_model(name, k, params, yhat, err=None):
        rss, aic, bic = aic_bic(y, yhat, k) if params is not None else (np.inf, np.inf, np.inf)
        models.append({"model": name, "rss": rss, "aic": aic, "bic": bic, "k": k, "params": None if params is None else list(map(float, params)), "error": err})
        if params is not None:
            preds[name] = yhat

    p = np.array([float(np.mean(y))]); add_model("constant", 1, p, constant_profile(Z, *p))
    p_lin = np.polyfit(Z, y, 1); p = np.array([p_lin[1], p_lin[0]]); add_model("linear", 2, p, linear_profile(Z, *p))

    p0 = [ymin, max(ymax - ymin, 1e-3), 1.0]
    p, _, err = _safe_curve_fit(pure_power_shifted_profile, Z, y, p0=p0, bounds=([-10, -20, 0.05], [10, 20, 8]))
    add_model("pure_power_shifted", 3, p, pure_power_shifted_profile(Z, *p) if p is not None else y, err)

    p0 = [ymin, ymax, zmid, width0]
    bounds = ([-5, -5, Z.min()-5, 0.05], [10, 10, Z.max()+5, 10])
    p, _, err = _safe_curve_fit(tanh_profile, Z, y, p0=p0, bounds=bounds)
    add_model("tanh", 4, p, tanh_profile(Z, *p) if p is not None else y, err)
    p, _, err = _safe_curve_fit(logistic_profile, Z, y, p0=p0, bounds=bounds)
    add_model("logistic", 4, p, logistic_profile(Z, *p) if p is not None else y, err)

    p0 = [ymin, ymax, zmid, 1.0 / width0]
    bounds = ([-5, -5, Z.min()-5, 0.05], [10, 10, Z.max()+5, 20])
    p, _, err = _safe_curve_fit(hill_saturating_profile, Z, y, p0=p0, bounds=bounds)
    add_model("hill_saturating_power", 4, p, hill_saturating_profile(Z, *p) if p is not None else y, err)

    fit_df = pd.DataFrame(models).replace([np.inf, -np.inf], np.nan)
    fit_df = fit_df.sort_values("aic", na_position="last").reset_index(drop=True)
    fit_df["delta_aic"] = fit_df["aic"] - fit_df["aic"].min()
    fit_df["delta_bic"] = fit_df["bic"] - fit_df["bic"].min()
    return fit_df, preds

def best_sigmoid_like(fit_df: pd.DataFrame) -> pd.Series:
    sig = fit_df[fit_df["model"].isin(["tanh", "logistic", "hill_saturating_power"])].dropna(subset=["aic"])
    if sig.empty:
        raise RuntimeError("No valid sigmoid-like model.")
    return sig.sort_values("aic").iloc[0]

def summarize_fit(tag: str, cfg: Cell4Config, agg: pd.DataFrame, fit_df: pd.DataFrame) -> Dict[str, Any]:
    sig = best_sigmoid_like(fit_df)
    const = fit_df[fit_df["model"] == "constant"].iloc[0]
    lin = fit_df[fit_df["model"] == "linear"].iloc[0]
    params = sig["params"]
    if sig["model"] == "hill_saturating_power":
        d_uv, d_ir, z_th, p = params
        Delta_Z = 1.0 / max(float(p), 1e-12)
    else:
        d_uv, d_ir, z_th, Delta_Z = params
    return {"tag": tag, **asdict(cfg), "best_aic_model": fit_df.iloc[0]["model"], "best_bic_model": fit_df.sort_values("bic").iloc[0]["model"],
            "best_sigmoid_like_model": sig["model"],
            "delta_aic_constant_vs_best_sigmoid": float(const["aic"] - sig["aic"]),
            "delta_bic_constant_vs_best_sigmoid": float(const["bic"] - sig["bic"]),
            "delta_aic_linear_vs_best_sigmoid": float(lin["aic"] - sig["aic"]),
            "delta_bic_linear_vs_best_sigmoid": float(lin["bic"] - sig["bic"]),
            "ds_low_mean_first3": float(agg["ds_mean"].head(3).mean()),
            "ds_high_mean_last3": float(agg["ds_mean"].tail(3).mean()),
            "ds_gain_high_minus_low": float(agg["ds_mean"].tail(3).mean() - agg["ds_mean"].head(3).mean()),
            "d_uv": float(d_uv), "d_ir": float(d_ir), "Z_th": float(z_th), "Delta_Z": float(Delta_Z),
            "tau_c_seconds_if_Z_is_physical": float(5.391247e-44 * math.exp(float(z_th)))}


In [ ]:
# ============================================================
# 8. Visualisation
# ============================================================

def plot_scan_and_fits(agg: pd.DataFrame, fit_df: pd.DataFrame, preds: Dict[str, np.ndarray], title: str = ""):
    Z = agg["Z"].to_numpy(); y = agg["ds_mean"].to_numpy(); yerr = agg["ds_sem"].to_numpy()
    plt.figure(figsize=(8, 5))
    plt.errorbar(Z, y, yerr=yerr, fmt="o", capsize=3, label="extracted $d_s^{eff}(Z)$")
    for name in ["constant", "linear", "pure_power_shifted", "tanh", "logistic", "hill_saturating_power"]:
        if name in preds:
            plt.plot(Z, preds[name], label=("Hill / saturating power" if name == "hill_saturating_power" else name), alpha=0.85)
    plt.xlabel("Z = ln(Δτ/tP)"); plt.ylabel("$d_s^{eff}$"); plt.title(title)
    plt.grid(True, alpha=0.3); plt.legend(); plt.show()

def plot_heat_curves_for_examples(curves: Dict[Tuple[float, int], Dict[str, Any]], n_examples: int = 4):
    keys = list(curves.keys())
    if not keys: return
    choose = [keys[i] for i in np.linspace(0, len(keys)-1, min(n_examples, len(keys))).astype(int)]
    plt.figure(figsize=(8, 5))
    for k in choose:
        c = curves[k]
        plt.loglog(c["u_grid"], c["P"], marker=".", label=f"Z={k[0]:.2f}, r={k[1]}")
    plt.xlabel("u"); plt.ylabel("P(u;Z)"); plt.title("Example heat traces")
    plt.grid(True, alpha=0.3); plt.legend(); plt.show()


In [ ]:
# ============================================================
# 9. Autotests légers
# ============================================================

def run_autotests():
    print("Running autotests...")
    small = replace(CFG, n_subdiv=2, layers=3, n_z=5, z_min=-1, z_max=3, n_realizations=2, heat_method="exact", n_probes=4, n_u=16, full_dense_limit=500, random_seed=777, conductance_rule="kigami_local")

    W = build_cell4_adjacency(small, Z=2.0, seed=1)
    Lp = laplacian_from_adjacency(W)
    D = W - W.T
    assert D.nnz == 0 or (D.data.size and np.max(np.abs(D.data)) < 1e-12), "Adjacency not symmetric."
    assert np.max(np.abs(np.asarray(Lp.sum(axis=1)).ravel())) < 1e-10, "Laplacian rows do not sum to zero."

    plane = replace(small, overlay_mode="plane2d", layers=1)
    Wp = build_cell4_adjacency(plane, Z=2.0, seed=2)
    assert count_vertical_edges(Wp, plane) == 0, "plane2d has vertical edges."
    assert np.all(np.isfinite(W.data)) and np.min(W.data) > 0, "Invalid weights."

    Z = np.linspace(-2, 4, 21)
    assert np.max(np.abs(hill_saturating_profile(Z, 2, 3, 1, 2) - logistic_profile(Z, 2, 3, 1, 1/2))) < 1e-12, "Hill/logistic equivalence failed."

    rng = np.random.default_rng(123)
    y = tanh_profile(Z, 2.0, 2.7, 1.1, 0.8) + 0.005 * rng.normal(size=len(Z))
    df = pd.DataFrame({"Z": Z, "ds_mean": y})
    fit_df, _ = fit_profile_models(df)
    sig = best_sigmoid_like(fit_df); const = fit_df[fit_df.model == "constant"].iloc[0]; lin = fit_df[fit_df.model == "linear"].iloc[0]
    assert const.aic - sig.aic > 10, "Synthetic sigmoid not preferred over constant."
    assert lin.aic - sig.aic > 6, "Synthetic sigmoid not preferred over linear."

    y0 = np.ones_like(Z) * 2.0
    fit0, _ = fit_profile_models(pd.DataFrame({"Z": Z, "ds_mean": y0}))
    assert fit0.iloc[0]["model"] in ["constant", "linear"], "Flat synthetic profile misread as sigmoid."

    print("All autotests passed.")
    return {"autotests_passed": True}

AUTOTEST_STATUS = {}
if RUN_AUTOTESTS:
    AUTOTEST_STATUS = run_autotests()


Running autotests...
All autotests passed.


In [ ]:
# ============================================================
# 10. Exécution complète avec sauvegarde incrémentale Drive
# ============================================================

def ensure_out_dir(path: str):
    if path:
        os.makedirs(path, exist_ok=True)


def _safe_listdir(path: str):
    try:
        return os.listdir(path)
    except Exception:
        return []


def _is_real_drive_mount(mount_point: str) -> bool:
    """Return True only if Colab Drive looks really mounted, not merely a local folder."""
    mydrive = os.path.join(mount_point, "MyDrive")
    return os.path.isdir(mydrive) and (os.path.ismount(mount_point) or os.path.exists("/content/drive/.shortcut-targets-by-id") or os.path.exists(mydrive))


def _prepare_clean_mountpoint(mount_point: str):
    """Avoid the common failure mode where /content/drive/MyDrive was created locally before mounting."""
    if os.path.exists(mount_point) and not os.path.ismount(mount_point):
        contents = _safe_listdir(mount_point)
        # If a fake /content/drive/MyDrive exists locally, mounting may silently fail or write locally.
        if contents:
            stamp = time.strftime("%Y%m%d_%H%M%S")
            backup = f"{mount_point}_LOCAL_BACKUP_{stamp}"
            print(f"[drive-setup] local non-mounted {mount_point} is not empty; moving it to {backup}")
            shutil.move(mount_point, backup)
    os.makedirs(mount_point, exist_ok=True)


def verify_drive_write(cfg: Cell4Config) -> bool:
    """Create a small marker file in the real Drive output folder and verify it can be read back."""
    try:
        ensure_out_dir(cfg.drive_out_dir)
        marker = os.path.join(cfg.drive_out_dir, "_DRIVE_MOUNT_OK.json")
        payload = {
            "status": "ok",
            "created_utc_like": time.strftime("%Y-%m-%d %H:%M:%S"),
            "drive_out_dir": cfg.drive_out_dir,
            "note": "If this file is visible in Google Drive, checkpoint export is active."
        }
        with open(marker, "w", encoding="utf-8") as f:
            json.dump(payload, f, indent=2)
        with open(marker, "r", encoding="utf-8") as f:
            check = json.load(f)
        ok = check.get("status") == "ok" and os.path.getsize(marker) > 0
        print("[drive-setup] marker written:", marker)
        print("[drive-setup] marker size:", os.path.getsize(marker), "bytes")
        return bool(ok)
    except Exception as exc:
        print(f"[drive-error] write verification failed: {exc}")
        return False


def setup_drive_and_output_dirs(cfg: Cell4Config):
    ensure_out_dir(cfg.out_dir)
    print("Local output:", cfg.out_dir)

    if not cfg.use_google_drive:
        print("Drive output: disabled")
        return

    try:
        from google.colab import drive
        _prepare_clean_mountpoint(cfg.drive_mount_point)
        print("[drive-setup] mounting Google Drive. If prompted, authorize access.")
        drive.mount(cfg.drive_mount_point, force_remount=True)
    except Exception as exc:
        print(f"[drive-error] Google Drive mount failed: {exc}")
        cfg.use_google_drive = False
        print("Drive output: disabled because mount failed")
        return

    mydrive = os.path.join(cfg.drive_mount_point, "MyDrive")
    if not os.path.isdir(mydrive):
        print(f"[drive-error] {mydrive} is not visible after mount. Drive is not ready.")
        cfg.use_google_drive = False
        return

    ensure_out_dir(cfg.drive_out_dir)
    if not verify_drive_write(cfg):
        print("[drive-error] Drive write test failed. Disabling Drive persistence for safety.")
        cfg.use_google_drive = False
        return

    print("Drive output:", cfg.drive_out_dir)
    print("Drive directory content after setup:", _safe_listdir(cfg.drive_out_dir))


setup_drive_and_output_dirs(CFG)

def save_json(obj: Dict[str, Any], path: str):
    def convert(o):
        if isinstance(o, (np.integer,)): return int(o)
        if isinstance(o, (np.floating,)): return float(o)
        if isinstance(o, np.ndarray): return o.tolist()
        if isinstance(o, (pd.Timestamp,)): return str(o)
        return str(o)
    ensure_out_dir(os.path.dirname(path))
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=convert)
    os.replace(tmp, path)

def _run_file_names(tag: str) -> List[str]:
    return [
        f"{tag}_raw.csv",
        f"{tag}_aggregated.csv",
        f"{tag}_fit_models.csv",
        f"{tag}_summary.json",
        f"{tag}_config.json",
        f"{tag}_PARTIAL_raw.csv",
    ]

def _path_pairs_for_tag(tag: str, cfg: Cell4Config) -> List[Tuple[str, str]]:
    return [(os.path.join(cfg.out_dir, name), os.path.join(cfg.drive_out_dir, name)) for name in _run_file_names(tag)]

def mirror_tag_to_drive(tag: str, cfg: Cell4Config):
    if not (cfg.use_google_drive and cfg.mirror_after_each_run):
        return
    ensure_out_dir(cfg.drive_out_dir)
    copied = 0
    for local, drive_path in _path_pairs_for_tag(tag, cfg):
        if os.path.exists(local):
            try:
                shutil.copy2(local, drive_path)
                copied += 1
            except Exception as exc:
                print(f"[drive-warning] could not copy {local} -> {drive_path}: {exc}")
    print(f"[drive] mirrored {copied} files for {tag}")

def restore_tag_from_drive(tag: str, cfg: Cell4Config):
    if not (cfg.use_google_drive and cfg.resume_from_drive):
        return
    ensure_out_dir(cfg.out_dir)
    restored = 0
    for local, drive_path in _path_pairs_for_tag(tag, cfg):
        if os.path.exists(drive_path) and not os.path.exists(local):
            try:
                shutil.copy2(drive_path, local)
                restored += 1
            except Exception as exc:
                print(f"[drive-warning] could not restore {drive_path} -> {local}: {exc}")
    if restored:
        print(f"[drive] restored {restored} files for {tag}")

def _complete_run_exists(tag: str, cfg: Cell4Config) -> bool:
    restore_tag_from_drive(tag, cfg)
    required = [f"{tag}_raw.csv", f"{tag}_aggregated.csv", f"{tag}_fit_models.csv", f"{tag}_summary.json", f"{tag}_config.json"]
    return all(os.path.exists(os.path.join(cfg.out_dir, name)) and os.path.getsize(os.path.join(cfg.out_dir, name)) > 0 for name in required)

def load_completed_run(tag: str, cfg: Cell4Config) -> Optional[Dict[str, Any]]:
    if not _complete_run_exists(tag, cfg):
        return None
    try:
        raw = pd.read_csv(os.path.join(cfg.out_dir, f"{tag}_raw.csv"))
        agg = pd.read_csv(os.path.join(cfg.out_dir, f"{tag}_aggregated.csv"))
        fit_df = pd.read_csv(os.path.join(cfg.out_dir, f"{tag}_fit_models.csv"))
        with open(os.path.join(cfg.out_dir, f"{tag}_summary.json"), "r", encoding="utf-8") as f:
            summary = json.load(f)
        return {"tag": tag, "cfg": cfg, "raw": raw, "agg": agg, "fit_df": fit_df, "preds": {}, "summary": summary, "curves": {}}
    except Exception as exc:
        print(f"[resume-warning] failed to load completed run {tag}: {exc}")
        return None

def write_run_index(cfg: Cell4Config):
    ensure_out_dir(cfg.out_dir)
    summaries = []
    for p in sorted(glob.glob(os.path.join(cfg.out_dir, "*_summary.json"))):
        try:
            with open(p, "r", encoding="utf-8") as f:
                summaries.append(json.load(f))
        except Exception:
            pass
    index = {"n_completed_summaries": len(summaries), "completed_tags": sorted([s.get("tag", Path(p).stem.replace("_summary", "")) for s, p in zip(summaries, glob.glob(os.path.join(cfg.out_dir, "*_summary.json")))]), "updated_at": time.strftime("%Y-%m-%d %H:%M:%S")}
    save_json(index, os.path.join(cfg.out_dir, "RUN_INDEX.json"))
    if cfg.use_google_drive and cfg.mirror_after_each_run:
        try:
            shutil.copy2(os.path.join(cfg.out_dir, "RUN_INDEX.json"), os.path.join(cfg.drive_out_dir, "RUN_INDEX.json"))
        except Exception as exc:
            print(f"[drive-warning] RUN_INDEX mirror failed: {exc}")

def zip_results(cfg: Cell4Config, suffix: str = "") -> str:
    ensure_out_dir(cfg.out_dir)
    base = cfg.out_dir.rstrip("/") + (suffix if suffix else "")
    zip_path = shutil.make_archive(base, "zip", cfg.out_dir)
    if cfg.use_google_drive:
        ensure_out_dir(cfg.drive_out_dir)
        drive_zip = os.path.join(cfg.drive_out_dir, os.path.basename(zip_path))
        try:
            shutil.copy2(zip_path, drive_zip)
            print("[drive] ZIP copied to:", drive_zip)
        except Exception as exc:
            print(f"[drive-warning] ZIP mirror failed: {exc}")
    return zip_path

def run_and_export(tag: str, cfg: Cell4Config, plot: bool = False) -> Dict[str, Any]:
    print("\n" + "="*72); print(f"[run] {tag}"); print(asdict(cfg)); t0 = time.time()

    if cfg.resume_from_drive and not cfg.force_rerun_completed:
        completed = load_completed_run(tag, cfg)
        if completed is not None:
            print(f"[resume] completed run found for {tag}; skipping computation.")
            return completed

    raw, curves = run_scan(cfg, verbose=True, checkpoint_tag=tag)
    expected = cfg.n_z * cfg.n_realizations
    if len(raw) < expected:
        print(f"[warning] {tag} incomplete: {len(raw)}/{expected} rows. Files are checkpointed; relaunch this cell later to continue.")

    agg = aggregate_scan(raw)
    fit_df, preds = fit_profile_models(agg, y_col="ds_mean")
    summary = summarize_fit(tag, cfg, agg, fit_df); summary["elapsed_s"] = float(time.time() - t0)
    summary["autotests_passed"] = bool(AUTOTEST_STATUS.get("autotests_passed", False)) if AUTOTEST_STATUS else None
    summary["n_raw_rows"] = int(len(raw))
    summary["n_raw_rows_expected"] = int(expected)
    summary["is_complete"] = bool(len(raw) == expected)

    ensure_out_dir(cfg.out_dir)
    raw.to_csv(os.path.join(cfg.out_dir, f"{tag}_raw.csv"), index=False)
    agg.to_csv(os.path.join(cfg.out_dir, f"{tag}_aggregated.csv"), index=False)
    fit_df.to_csv(os.path.join(cfg.out_dir, f"{tag}_fit_models.csv"), index=False)
    save_json(summary, os.path.join(cfg.out_dir, f"{tag}_summary.json"))
    save_json(asdict(cfg), os.path.join(cfg.out_dir, f"{tag}_config.json"))

    mirror_tag_to_drive(tag, cfg)
    write_run_index(cfg)
    if cfg.zip_after_each_run:
        zip_results(cfg, suffix="_checkpoint")

    print("Summary:")
    print(json.dumps({k: summary[k] for k in ["tag", "n_realizations", "conductance_rule", "best_aic_model", "best_sigmoid_like_model", "delta_aic_constant_vs_best_sigmoid", "delta_aic_linear_vs_best_sigmoid", "d_uv", "d_ir", "Z_th", "Delta_Z", "n_raw_rows", "is_complete"] if k in summary}, indent=2))
    display(agg); display(fit_df)
    if plot:
        plot_scan_and_fits(agg, fit_df, preds, title=tag)
        if curves:
            plot_heat_curves_for_examples(curves, n_examples=4)
    return {"tag": tag, "cfg": cfg, "raw": raw, "agg": agg, "fit_df": fit_df, "preds": preds, "summary": summary, "curves": curves}


Local output: /content/cell4_test1K_kigami_outputs
[drive-setup] mounting Google Drive. If prompted, authorize access.
Mounted at /content/drive
[drive-setup] marker written: /content/drive/MyDrive/DFGQ_Test1K/cell4_test1K_kigami_outputs/_DRIVE_MOUNT_OK.json
[drive-setup] marker size: 233 bytes
Drive output: /content/drive/MyDrive/DFGQ_Test1K/cell4_test1K_kigami_outputs
Drive directory content after setup: ['_DRIVE_MOUNT_OK.json']


In [ ]:
# ============================================================
# 11. Run principal + contrôles
# ============================================================

runs = []

if RUN_NEGATIVE_CONTROL:
    plane_cfg = replace(CFG, overlay_mode="plane2d", layers=1, conductance_rule="phenomenological", random_seed=2000)
    runs.append(run_and_export("negative_plane2d", plane_cfg, plot=PLOT_EACH_VARIANT))

if RUN_MAIN:
    main_cfg = replace(CFG, conductance_rule="kigami_local", random_seed=2100)
    runs.append(run_and_export("main_kigami_local", main_cfg, plot=PLOT_EACH_VARIANT))

if RUN_PHENOM_COMPARISON:
    phenom_cfg = replace(CFG, conductance_rule="phenomenological", random_seed=2200)
    runs.append(run_and_export("comparison_phenomenological", phenom_cfg, plot=PLOT_EACH_VARIANT))

main_summary_df = pd.DataFrame([r["summary"] for r in runs])
if len(main_summary_df):
    cols = ["tag", "conductance_rule", "overlay_mode", "layers", "n_realizations", "n_raw_rows", "is_complete", "best_aic_model", "best_sigmoid_like_model", "delta_aic_constant_vs_best_sigmoid", "delta_aic_linear_vs_best_sigmoid", "d_uv", "d_ir", "Z_th", "Delta_Z", "tau_c_seconds_if_Z_is_physical"]
    display(main_summary_df[[c for c in cols if c in main_summary_df.columns]])
    main_summary_df.to_csv(os.path.join(CFG.out_dir, "main_runs_summary.csv"), index=False)
    if CFG.use_google_drive:
        shutil.copy2(os.path.join(CFG.out_dir, "main_runs_summary.csv"), os.path.join(CFG.drive_out_dir, "main_runs_summary.csv"))



[run] negative_plane2d
{'n_subdiv': 4, 'layers': 1, 'overlay_mode': 'plane2d', 'torus_xy': True, 'torus_layers': False, 'z_min': -1.0, 'z_max': 5.0, 'n_z': 17, 'theta_center': 2.0, 'theta_width_hier': 1.1, 'jitter': 0.35, 'w_plane': 1.0, 'w_vertical_lo': 0.0, 'w_vertical_hi': 1.0, 'conductance_rule': 'phenomenological', 'resistance_scale': 1.0, 'resistance_hier_power': 1.0, 'resistance_eps': 1e-12, 'normalize_active_vertical_mean': True, 'heat_method': 'hutchinson', 'n_probes': 24, 'u_min': 0.08, 'u_max': 80.0, 'n_u': 34, 'full_dense_limit': 700, 'n_realizations': 50, 'random_seed': 2000, 'p_high': 0.85, 'p_low_floor': 0.015, 'min_window_points': 6, 'smooth_ds': True, 'savgol_window': 7, 'savgol_poly': 2, 'out_dir': '/content/cell4_test1K_kigami_outputs', 'use_google_drive': True, 'drive_mount_point': '/content/drive', 'drive_out_dir': '/content/drive/MyDrive/DFGQ_Test1K/cell4_test1K_kigami_outputs', 'resume_from_drive': True, 'force_rerun_completed': False, 'checkpoint_after_each_Z':

,Z,ds_mean,ds_median,ds_std,ds_sem,lcc_ratio,n_components,E_undirected,E_vertical,window_points,n_rows
0,-1.000,1.801135,2.099990,0.038951,0.005509,1.0,1.0,512.0,0.0,19.44,50
1,-0.625,1.802442,2.100751,0.040320,0.005702,1.0,1.0,512.0,0.0,19.40,50
2,-0.250,1.800407,2.099084,0.034694,0.004906,1.0,1.0,512.0,0.0,19.42,50
3,0.125,1.794705,2.098187,0.025475,0.003603,1.0,1.0,512.0,0.0,19.52,50
4,0.500,1.806409,2.120022,0.043487,0.006150,1.0,1.0,512.0,0.0,19.34,50
5,0.875,1.791774,2.095808,0.035173,0.004974,1.0,1.0,512.0,0.0,19.50,50
6,1.250,1.795583,2.084994,0.039870,0.005638,1.0,1.0,512.0,0.0,19.46,50
7,1.625,1.798774,2.098382,0.043450,0.006145,1.0,1.0,512.0,0.0,19.44,50
8,2.000,1.790523,2.076320,0.042737,0.006044,1.0,1.0,512.0,0.0,19.50,50
9,2.375,1.798493,2.097763,0.039513,0.005588,1.0,1.0,512.0,0.0,19.42,50


,model,rss,aic,bic,k,params,error,delta_aic,delta_bic
0,constant,0.000295,-184.362811,-183.529598,1,[1.7982991802977686],None,0.000000,0.000000
1,linear,0.000290,-182.626634,-180.960208,2,"[1.7988617311408404, -0.00028127542153564585]",None,1.736177,2.569390
2,pure_power_shifted,0.000280,-181.221907,-178.722267,3,"[1.8016470443936865, -0.0029146250406961965, 0...",None,3.140904,4.807331
3,hill_saturating_power,0.000260,-180.489118,-177.156264,4,"[1.8017916209864464, 1.7976504555937671, -0.21...",None,3.873694,6.373334
4,logistic,0.000260,-180.489118,-177.156264,4,"[1.8017916219494892, 1.797650455663901, -0.216...",None,3.873694,6.373334
5,tanh,0.000284,-179.025276,-175.692422,4,"[1.7985827704091215, 1.7954609339812906, 4.620...",None,5.337535,7.837175



[run] main_kigami_local
{'n_subdiv': 4, 'layers': 8, 'overlay_mode': 'multiplex', 'torus_xy': True, 'torus_layers': False, 'z_min': -1.0, 'z_max': 5.0, 'n_z': 17, 'theta_center': 2.0, 'theta_width_hier': 1.1, 'jitter': 0.35, 'w_plane': 1.0, 'w_vertical_lo': 0.0, 'w_vertical_hi': 1.0, 'conductance_rule': 'kigami_local', 'resistance_scale': 1.0, 'resistance_hier_power': 1.0, 'resistance_eps': 1e-12, 'normalize_active_vertical_mean': True, 'heat_method': 'hutchinson', 'n_probes': 24, 'u_min': 0.08, 'u_max': 80.0, 'n_u': 34, 'full_dense_limit': 700, 'n_realizations': 50, 'random_seed': 2100, 'p_high': 0.85, 'p_low_floor': 0.015, 'min_window_points': 6, 'smooth_ds': True, 'savgol_window': 7, 'savgol_poly': 2, 'out_dir': '/content/cell4_test1K_kigami_outputs', 'use_google_drive': True, 'drive_mount_point': '/content/drive', 'drive_out_dir': '/content/drive/MyDrive/DFGQ_Test1K/cell4_test1K_kigami_outputs', 'resume_from_drive': True, 'force_rerun_completed': False, 'checkpoint_after_each_Z': 

,Z,ds_mean,ds_median,ds_std,ds_sem,lcc_ratio,n_components,E_undirected,E_vertical,window_points,n_rows
0,-1.000,1.818348,2.090087,0.013922,0.001969,0.1250,8.00,4096.00,0.00,20.74,50
1,-0.625,1.812977,2.070150,0.013915,0.001968,0.1250,8.00,4096.00,0.00,20.90,50
2,-0.250,1.816706,2.079987,0.012035,0.001702,0.1250,8.00,4096.00,0.00,20.86,50
3,0.125,1.817163,2.084271,0.014284,0.002020,0.1325,7.94,4096.06,0.06,20.72,50
4,0.500,1.816864,2.081386,0.013446,0.001902,0.1950,7.34,4096.68,0.68,20.76,50
5,0.875,1.827022,2.128047,0.011976,0.001694,0.7700,1.98,4109.32,13.32,20.46,50
6,1.250,1.893124,2.301179,0.015896,0.002248,1.0000,1.00,4183.42,87.42,20.00,50
7,1.625,2.005224,2.365104,0.015711,0.002222,1.0000,1.00,4394.64,298.64,18.88,50
8,2.000,2.128277,2.344324,0.018366,0.002597,1.0000,1.00,4787.76,691.76,17.00,50
9,2.375,2.279170,2.441325,0.017510,0.002476,1.0000,1.00,5310.60,1214.60,16.00,50


,model,rss,aic,bic,k,params,error,delta_aic,delta_bic
0,logistic,0.000800,-161.389361,-158.056508,4,"[1.8126935105104638, 2.38287199663533, 1.88781...",None,0.000000e+00,0.000000e+00
1,tanh,0.000800,-161.389361,-158.056508,4,"[1.8126934854562962, 2.382872027926583, 1.8878...",None,3.451248e-10,3.451248e-10
2,hill_saturating_power,0.000800,-161.389361,-158.056508,4,"[1.812693477634876, 2.3828720371967114, 1.8878...",None,6.730545e-10,6.730545e-10
3,linear,0.133677,-78.374217,-76.707790,2,"[1.8500363004525306, 0.12888918011480366]",None,8.301514e+01,8.134872e+01
4,pure_power_shifted,0.132652,-76.505102,-74.005462,3,"[1.7368289720839705, 0.11229554349915885, 1.07...",None,8.488426e+01,8.405105e+01
5,constant,1.086815,-44.749359,-43.916145,1,[2.1078146606821377],None,1.166400e+02,1.141404e+02



[run] comparison_phenomenological
{'n_subdiv': 4, 'layers': 8, 'overlay_mode': 'multiplex', 'torus_xy': True, 'torus_layers': False, 'z_min': -1.0, 'z_max': 5.0, 'n_z': 17, 'theta_center': 2.0, 'theta_width_hier': 1.1, 'jitter': 0.35, 'w_plane': 1.0, 'w_vertical_lo': 0.0, 'w_vertical_hi': 1.0, 'conductance_rule': 'phenomenological', 'resistance_scale': 1.0, 'resistance_hier_power': 1.0, 'resistance_eps': 1e-12, 'normalize_active_vertical_mean': True, 'heat_method': 'hutchinson', 'n_probes': 24, 'u_min': 0.08, 'u_max': 80.0, 'n_u': 34, 'full_dense_limit': 700, 'n_realizations': 50, 'random_seed': 2200, 'p_high': 0.85, 'p_low_floor': 0.015, 'min_window_points': 6, 'smooth_ds': True, 'savgol_window': 7, 'savgol_poly': 2, 'out_dir': '/content/cell4_test1K_kigami_outputs', 'use_google_drive': True, 'drive_mount_point': '/content/drive', 'drive_out_dir': '/content/drive/MyDrive/DFGQ_Test1K/cell4_test1K_kigami_outputs', 'resume_from_drive': True, 'force_rerun_completed': False, 'checkpoint_a

,Z,ds_mean,ds_median,ds_std,ds_sem,lcc_ratio,n_components,E_undirected,E_vertical,window_points,n_rows
0,-1.000,1.818031,2.088039,0.013396,0.001894,0.1250,8.00,4096.00,0.00,20.70,50
1,-0.625,1.816447,2.078854,0.012030,0.001701,0.1250,8.00,4096.00,0.00,20.82,50
2,-0.250,1.814614,2.074683,0.014020,0.001983,0.1250,8.00,4096.00,0.00,20.90,50
3,0.125,1.818211,2.082156,0.014821,0.002096,0.1250,8.00,4096.00,0.00,20.86,50
4,0.500,1.819521,2.084872,0.013802,0.001952,0.1925,7.26,4096.76,0.76,20.78,50
5,0.875,1.826308,2.123048,0.010544,0.001491,0.7750,2.00,4108.76,12.76,20.50,50
6,1.250,1.888852,2.294492,0.017106,0.002419,1.0000,1.00,4182.10,86.10,20.00,50
7,1.625,2.008004,2.354937,0.013969,0.001976,1.0000,1.00,4395.50,299.50,18.70,50
8,2.000,2.151042,2.371306,0.020412,0.002887,1.0000,1.00,4789.30,693.30,17.00,50
9,2.375,2.294139,2.490717,0.016472,0.002330,1.0000,1.00,5299.92,1203.92,16.00,50


,model,rss,aic,bic,k,params,error,delta_aic,delta_bic
0,tanh,0.000493,-169.611501,-166.278647,4,"[1.812540760373276, 2.407040218029548, 1.89224...",None,0.000000e+00,0.000000e+00
1,hill_saturating_power,0.000493,-169.611501,-166.278647,4,"[1.8125407483223868, 2.407040224553126, 1.8922...",None,1.555804e-10,1.555804e-10
2,logistic,0.000493,-169.611501,-166.278647,4,"[1.8125407387519532, 2.4070402300522433, 1.892...",None,4.242509e-10,4.242509e-10
3,linear,0.142881,-77.242287,-75.575860,2,"[1.8506492376130443, 0.13459428951203012]",None,9.236921e+01,9.070279e+01
4,pure_power_shifted,0.141403,-75.419048,-72.919408,3,"[1.7348218501879227, 0.11476881254078852, 1.08...",None,9.419245e+01,9.335924e+01
5,constant,1.182265,-43.318287,-42.485073,1,[2.119837816637105],None,1.262932e+02,1.237936e+02


,tag,conductance_rule,overlay_mode,layers,n_realizations,n_raw_rows,is_complete,best_aic_model,best_sigmoid_like_model,delta_aic_constant_vs_best_sigmoid,delta_aic_linear_vs_best_sigmoid,d_uv,d_ir,Z_th,Delta_Z,tau_c_seconds_if_Z_is_physical
0,negative_plane2d,phenomenological,plane2d,1,50,850,True,constant,hill_saturating_power,-3.873694,-2.137517,1.801792,1.797650,-0.216323,0.050000,4.342514e-44
1,main_kigami_local,kigami_local,multiplex,8,50,850,True,logistic,logistic,116.640003,83.015144,1.812694,2.382872,1.887818,0.339138,3.560888e-43
2,comparison_phenomenological,phenomenological,multiplex,8,50,850,True,tanh,tanh,126.293214,92.369213,1.812541,2.407040,1.892243,0.676173,3.576678e-43


In [ ]:
# ============================================================
# 12. Robustesse rapide, relançable depuis Drive
# ============================================================

robustness_runs = []

def fast_robustness_variants(base: Cell4Config) -> List[Tuple[str, Cell4Config]]:
    return [
        ("base_kigami_p24", replace(base, n_probes=24, random_seed=3100)),
        ("torus_layers_true", replace(base, torus_layers=True, n_probes=24, random_seed=3200)),
        ("layers12_torus", replace(base, layers=12, torus_layers=True, n_probes=24, random_seed=3300)),
        ("vertical_hi_1p5", replace(base, w_vertical_hi=1.5, torus_layers=True, n_probes=24, random_seed=3400)),
        ("wider_Z_grid", replace(base, z_min=-1.5, z_max=5.5, n_z=19, n_probes=24, random_seed=3500)),
    ]

if RUN_FAST_ROBUSTNESS:
    variants = fast_robustness_variants(CFG)
    print("Number of fast robustness variants:", len(variants))
    for tag, cfgv in variants:
        robustness_runs.append(run_and_export(tag, cfgv, plot=PLOT_EACH_VARIANT))

print("Fast robustness loaded/computed:", len(robustness_runs))


Number of fast robustness variants: 5

[run] base_kigami_p24
{'n_subdiv': 4, 'layers': 8, 'overlay_mode': 'multiplex', 'torus_xy': True, 'torus_layers': False, 'z_min': -1.0, 'z_max': 5.0, 'n_z': 17, 'theta_center': 2.0, 'theta_width_hier': 1.1, 'jitter': 0.35, 'w_plane': 1.0, 'w_vertical_lo': 0.0, 'w_vertical_hi': 1.0, 'conductance_rule': 'kigami_local', 'resistance_scale': 1.0, 'resistance_hier_power': 1.0, 'resistance_eps': 1e-12, 'normalize_active_vertical_mean': True, 'heat_method': 'hutchinson', 'n_probes': 24, 'u_min': 0.08, 'u_max': 80.0, 'n_u': 34, 'full_dense_limit': 700, 'n_realizations': 50, 'random_seed': 3100, 'p_high': 0.85, 'p_low_floor': 0.015, 'min_window_points': 6, 'smooth_ds': True, 'savgol_window': 7, 'savgol_poly': 2, 'out_dir': '/content/cell4_test1K_kigami_outputs', 'use_google_drive': True, 'drive_mount_point': '/content/drive', 'drive_out_dir': '/content/drive/MyDrive/DFGQ_Test1K/cell4_test1K_kigami_outputs', 'resume_from_drive': True, 'force_rerun_completed'

,Z,ds_mean,ds_median,ds_std,ds_sem,lcc_ratio,n_components,E_undirected,E_vertical,window_points,n_rows
0,-1.000,1.814547,2.079775,0.014079,0.001991,0.1250,8.00,4096.00,0.00,20.82,50
1,-0.625,1.813944,2.078330,0.014470,0.002046,0.1250,8.00,4096.00,0.00,20.80,50
2,-0.250,1.817520,2.081152,0.013913,0.001968,0.1250,8.00,4096.00,0.00,20.76,50
3,0.125,1.817769,2.085134,0.013571,0.001919,0.1300,7.96,4096.04,0.00,20.86,50
4,0.500,1.816310,2.082744,0.012899,0.001824,0.2225,7.02,4097.02,0.90,20.80,50
5,0.875,1.830675,2.138175,0.010451,0.001478,0.8700,1.70,4109.88,12.08,20.32,50
6,1.250,1.899421,2.309351,0.019574,0.002768,1.0000,1.00,4194.98,87.04,19.92,50
7,1.625,2.013145,2.326126,0.012245,0.001732,1.0000,1.00,4439.46,300.50,18.24,50
8,2.000,2.193140,2.384874,0.016985,0.002402,1.0000,1.00,4889.44,693.42,17.00,50
9,2.375,2.369255,2.505542,0.018402,0.002602,1.0000,1.00,5472.04,1204.16,15.98,50


,model,rss,aic,bic,k,params,error,delta_aic,delta_bic
0,hill_saturating_power,0.000534,-168.259110,-164.926256,4,"[1.8135086346391303, 2.501484410175927, 1.9193...",None,0.000000e+00,0.000000e+00
1,tanh,0.000534,-168.259110,-164.926256,4,"[1.8135086317383522, 2.50148441573531, 1.91935...",None,5.073275e-11,5.073275e-11
2,logistic,0.000534,-168.259110,-164.926256,4,"[1.8135086487522976, 2.5014843846653876, 1.919...",None,3.666116e-10,3.666116e-10
3,linear,0.201557,-71.393260,-69.726833,2,"[1.8534848050115083, 0.15635697982800453]",None,9.686585e+01,9.519942e+01
4,pure_power_shifted,0.199410,-69.575293,-67.075653,3,"[1.7197304014866026, 0.13249113629899203, 1.08...",None,9.868382e+01,9.785060e+01
5,constant,1.604232,-38.129656,-37.296442,1,[2.166198764667517],None,1.301295e+02,1.276298e+02



[run] layers12_torus
{'n_subdiv': 4, 'layers': 12, 'overlay_mode': 'multiplex', 'torus_xy': True, 'torus_layers': True, 'z_min': -1.0, 'z_max': 5.0, 'n_z': 17, 'theta_center': 2.0, 'theta_width_hier': 1.1, 'jitter': 0.35, 'w_plane': 1.0, 'w_vertical_lo': 0.0, 'w_vertical_hi': 1.0, 'conductance_rule': 'kigami_local', 'resistance_scale': 1.0, 'resistance_hier_power': 1.0, 'resistance_eps': 1e-12, 'normalize_active_vertical_mean': True, 'heat_method': 'hutchinson', 'n_probes': 24, 'u_min': 0.08, 'u_max': 80.0, 'n_u': 34, 'full_dense_limit': 700, 'n_realizations': 50, 'random_seed': 3300, 'p_high': 0.85, 'p_low_floor': 0.015, 'min_window_points': 6, 'smooth_ds': True, 'savgol_window': 7, 'savgol_poly': 2, 'out_dir': '/content/cell4_test1K_kigami_outputs', 'use_google_drive': True, 'drive_mount_point': '/content/drive', 'drive_out_dir': '/content/drive/MyDrive/DFGQ_Test1K/cell4_test1K_kigami_outputs', 'resume_from_drive': True, 'force_rerun_completed': False, 'checkpoint_after_each_Z': Tru

,Z,ds_mean,ds_median,ds_std,ds_sem,lcc_ratio,n_components,E_undirected,E_vertical,window_points,n_rows
0,-1.000,1.817264,2.083911,0.009573,0.001354,0.083333,12.00,6144.00,0.00,20.84,50
1,-0.625,1.815777,2.079068,0.010562,0.001494,0.083333,12.00,6144.00,0.00,20.88,50
2,-0.250,1.815485,2.078469,0.010326,0.001460,0.083333,12.00,6144.00,0.00,20.88,50
3,0.125,1.816165,2.077321,0.012566,0.001777,0.086667,11.96,6144.04,0.04,20.88,50
4,0.500,1.813613,2.073479,0.013679,0.001934,0.151667,10.82,6145.20,1.12,20.92,50
5,0.875,1.827897,2.130344,0.011198,0.001584,0.823333,1.82,6165.84,19.82,20.46,50
6,1.250,1.899549,2.312573,0.013932,0.001970,1.000000,1.00,6291.14,134.70,19.98,50
7,1.625,2.012277,2.328404,0.011619,0.001643,1.000000,1.00,6660.58,473.66,18.28,50
8,2.000,2.186365,2.381136,0.013437,0.001900,1.000000,1.00,7328.40,1085.24,17.00,50
9,2.375,2.371188,2.504597,0.016738,0.002367,1.000000,1.00,8209.78,1893.52,15.96,50


,model,rss,aic,bic,k,params,error,delta_aic,delta_bic
0,logistic,0.000760,-162.253051,-158.920198,4,"[1.8132932344057122, 2.50464839667431, 1.92809...",None,0.000000e+00,0.000000e+00
1,tanh,0.000760,-162.253051,-158.920198,4,"[1.8132932144969218, 2.504648434619977, 1.9280...",None,8.409984e-11,8.409984e-11
2,hill_saturating_power,0.000760,-162.253051,-158.920198,4,"[1.8132932023642496, 2.5046484560920503, 1.928...",None,8.225527e-10,8.225527e-10
3,linear,0.207288,-70.916595,-69.250169,2,"[1.8528282843137438, 0.1569696336097327]",None,9.133646e+01,8.967003e+01
4,pure_power_shifted,0.204786,-69.123039,-66.623399,3,"[1.7201131117974549, 0.13138156728860703, 1.09...",None,9.313001e+01,9.229680e+01
5,constant,1.620978,-37.953127,-37.119914,1,[2.166767551533209],None,1.242999e+02,1.218003e+02



[run] vertical_hi_1p5
{'n_subdiv': 4, 'layers': 8, 'overlay_mode': 'multiplex', 'torus_xy': True, 'torus_layers': True, 'z_min': -1.0, 'z_max': 5.0, 'n_z': 17, 'theta_center': 2.0, 'theta_width_hier': 1.1, 'jitter': 0.35, 'w_plane': 1.0, 'w_vertical_lo': 0.0, 'w_vertical_hi': 1.5, 'conductance_rule': 'kigami_local', 'resistance_scale': 1.0, 'resistance_hier_power': 1.0, 'resistance_eps': 1e-12, 'normalize_active_vertical_mean': True, 'heat_method': 'hutchinson', 'n_probes': 24, 'u_min': 0.08, 'u_max': 80.0, 'n_u': 34, 'full_dense_limit': 700, 'n_realizations': 50, 'random_seed': 3400, 'p_high': 0.85, 'p_low_floor': 0.015, 'min_window_points': 6, 'smooth_ds': True, 'savgol_window': 7, 'savgol_poly': 2, 'out_dir': '/content/cell4_test1K_kigami_outputs', 'use_google_drive': True, 'drive_mount_point': '/content/drive', 'drive_out_dir': '/content/drive/MyDrive/DFGQ_Test1K/cell4_test1K_kigami_outputs', 'resume_from_drive': True, 'force_rerun_completed': False, 'checkpoint_after_each_Z': Tru

In [ ]:
# ============================================================
# 13. Publication sweep, relançable depuis Drive
# ============================================================

def publication_sweep_variants(base: Cell4Config) -> List[Tuple[str, Cell4Config]]:
    variants = []; seed = 5000
    for layers in [8, 12, 16]:
        for torus_layers in [False, True]:
            for n_probes in [32, 64]:
                tag = f"pub_layers{layers}_torus{int(torus_layers)}_p{n_probes}"
                variants.append((tag, replace(base, layers=layers, torus_layers=torus_layers, n_probes=n_probes, z_min=-1.5, z_max=5.5, n_z=21, random_seed=seed)))
                seed += 100
    return variants

if RUN_PUBLICATION_SWEEP:
    variants = publication_sweep_variants(CFG)
    print("Number of publication sweep variants:", len(variants))
    for tag, cfgv in variants:
        robustness_runs.append(run_and_export(tag, cfgv, plot=PLOT_EACH_VARIANT))

print("Total robustness loaded/computed:", len(robustness_runs))


NameError: name 'Cell4Config' is not defined

In [ ]:
# ============================================================
# 14. Synthèse de robustesse depuis mémoire + Drive
# ============================================================

def expected_robustness_variants(base: Cell4Config) -> List[Tuple[str, Cell4Config]]:
    variants: List[Tuple[str, Cell4Config]] = []
    if RUN_FAST_ROBUSTNESS:
        variants.extend(fast_robustness_variants(base))
    if RUN_PUBLICATION_SWEEP:
        variants.extend(publication_sweep_variants(base))
    return variants

def merge_current_and_disk_runs(current: List[Dict[str, Any]], expected: List[Tuple[str, Cell4Config]]) -> List[Dict[str, Any]]:
    by_tag = {r["tag"]: r for r in current if isinstance(r, dict) and "tag" in r}
    for tag, cfgv in expected:
        if tag not in by_tag:
            loaded = load_completed_run(tag, cfgv)
            if loaded is not None:
                by_tag[tag] = loaded
    return [by_tag[tag] for tag, _ in expected if tag in by_tag]

def summarize_robustness(run_list: List[Dict[str, Any]], out_dir: str) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    if not run_list:
        return pd.DataFrame(), {"n_variants": 0, "reviewer_safe_minimum": False}
    df = pd.DataFrame([r["summary"] for r in run_list])
    complete_mask = df.get("is_complete", pd.Series([True]*len(df))).fillna(True).astype(bool)
    df_complete = df[complete_mask].copy()
    if len(df_complete) == 0:
        df_complete = df.copy()
    frac_sig_const_aic = float(np.mean(df_complete["delta_aic_constant_vs_best_sigmoid"] > 10))
    frac_sig_const_bic = float(np.mean(df_complete["delta_bic_constant_vs_best_sigmoid"] > 6))
    frac_lin_rej_aic = float(np.mean(df_complete["delta_aic_linear_vs_best_sigmoid"] > 6))
    summary = {
        "n_variants_loaded": int(len(df)),
        "n_variants_complete": int(len(df_complete)),
        "n_variants_expected": int(len(expected_robustness_variants(CFG))),
        "fraction_sigmoid_AIC_vs_constant_delta_gt_10": frac_sig_const_aic,
        "fraction_sigmoid_BIC_vs_constant_delta_gt_6": frac_sig_const_bic,
        "fraction_linear_rejected_delta_AIC_gt_6": frac_lin_rej_aic,
        "Z_th_mean": float(df_complete["Z_th"].mean()), "Z_th_std": float(df_complete["Z_th"].std(ddof=1)) if len(df_complete)>1 else 0.0,
        "Delta_Z_mean": float(df_complete["Delta_Z"].mean()), "Delta_Z_std": float(df_complete["Delta_Z"].std(ddof=1)) if len(df_complete)>1 else 0.0,
        "d_uv_mean": float(df_complete["d_uv"].mean()), "d_uv_std": float(df_complete["d_uv"].std(ddof=1)) if len(df_complete)>1 else 0.0,
        "d_ir_mean": float(df_complete["d_ir"].mean()), "d_ir_std": float(df_complete["d_ir"].std(ddof=1)) if len(df_complete)>1 else 0.0,
        "reviewer_safe_minimum": bool(frac_sig_const_aic == 1.0 and frac_sig_const_bic == 1.0 and frac_lin_rej_aic == 1.0 and len(df_complete) == len(expected_robustness_variants(CFG)))
    }
    ensure_out_dir(out_dir)
    df.to_csv(os.path.join(out_dir, "robustness_summary_all_loaded.csv"), index=False)
    df_complete.to_csv(os.path.join(out_dir, "robustness_summary.csv"), index=False)
    save_json(summary, os.path.join(out_dir, "robustness_global_summary.json"))
    if CFG.use_google_drive:
        for name in ["robustness_summary_all_loaded.csv", "robustness_summary.csv", "robustness_global_summary.json"]:
            try:
                shutil.copy2(os.path.join(out_dir, name), os.path.join(CFG.drive_out_dir, name))
            except Exception as exc:
                print(f"[drive-warning] could not mirror {name}: {exc}")
    return df, summary

expected_variants = expected_robustness_variants(CFG)
robustness_runs = merge_current_and_disk_runs(robustness_runs, expected_variants)
robustness_df, robustness_summary = summarize_robustness(robustness_runs, CFG.out_dir)

if len(robustness_df):
    cols = ["tag", "conductance_rule", "layers", "torus_layers", "n_realizations", "n_probes", "n_raw_rows", "n_raw_rows_expected", "is_complete", "best_aic_model", "best_sigmoid_like_model", "delta_aic_constant_vs_best_sigmoid", "delta_aic_linear_vs_best_sigmoid", "d_uv", "d_ir", "Z_th", "Delta_Z"]
    display(robustness_df[[c for c in cols if c in robustness_df.columns]])
print(json.dumps(robustness_summary, indent=2))
print(f"Loaded/completed variants: {robustness_summary.get('n_variants_loaded', 0)}/{robustness_summary.get('n_variants_expected', len(expected_variants))}")


In [ ]:
# ============================================================
# 15. Figures consolidées
# ============================================================

if len(robustness_df):
    for col, label, fname in [("Z_th", "$Z_{th}$", "robustness_Z_th.png"), ("Delta_Z", "$\\Delta Z$", "robustness_Delta_Z.png"), ("d_ir", "$d_{IR}$", "robustness_d_ir.png")]:
        plt.figure(figsize=(8, 4.5))
        plt.errorbar(np.arange(len(robustness_df)), robustness_df[col], yerr=0, fmt="o")
        plt.axhline(float(robustness_df[col].mean()), linestyle="--", label=f"mean={float(robustness_df[col].mean()):.3f}")
        plt.xticks(np.arange(len(robustness_df)), robustness_df["tag"], rotation=60, ha="right")
        plt.ylabel(label); plt.title(f"Robustness of {col}")
        plt.grid(True, alpha=0.3); plt.legend(); plt.tight_layout()
        plt.savefig(os.path.join(CFG.out_dir, fname), dpi=160)
        plt.show()


In [ ]:
# ============================================================
# 16. Rapport reviewer-safe et export ZIP local + Drive
# ============================================================

report = {
    "title": "Test 1K — Kigami-weighted cell-4 spectral crossover",
    "status": "completed_or_checkpointed_notebook_run" if (len(runs) or len(robustness_runs)) else "not_run",
    "configuration": asdict(CFG),
    "autotests": AUTOTEST_STATUS,
    "main_runs": [r["summary"] for r in runs],
    "robustness_global_summary": robustness_summary,
    "drive_persistence": {
        "local_out_dir": CFG.out_dir,
        "drive_out_dir": CFG.drive_out_dir,
        "resume_from_drive": CFG.resume_from_drive,
        "checkpoint_after_each_Z": CFG.checkpoint_after_each_Z,
        "force_rerun_completed": CFG.force_rerun_completed,
    },
    "interpretation": {
        "safe_claim": "The test upgrades the original cell-4 spectral crossover into a Dirichlet-energy-based numerical protocol by replacing vertical phenomenological weights with local resistance-conductance proxies.",
        "not_claimed": "This is not a proof of Einstein gravity, electromagnetic coupling, or a closed analytical Kigami derivation.",
        "hill_note": "The Hill/saturating-power competitor is included because a bounded power law in x=exp(Z) is equivalent to a logistic crossover in Z."
    }
}

ensure_out_dir(CFG.out_dir)
save_json(report, os.path.join(CFG.out_dir, "TEST1K_REPORT.json"))
save_json({"manifest": sorted(os.listdir(CFG.out_dir))}, os.path.join(CFG.out_dir, "ZIP_MANIFEST.json"))
write_run_index(CFG)

if CFG.use_google_drive:
    for name in ["TEST1K_REPORT.json", "ZIP_MANIFEST.json", "RUN_INDEX.json"]:
        try:
            shutil.copy2(os.path.join(CFG.out_dir, name), os.path.join(CFG.drive_out_dir, name))
        except Exception as exc:
            print(f"[drive-warning] could not mirror {name}: {exc}")

zip_path = zip_results(CFG)
print("Exports written locally to:", CFG.out_dir)
print("Exports mirrored to Drive:", CFG.drive_out_dir if CFG.use_google_drive else "disabled")
print("ZIP written locally to:", zip_path)
print("Current output files:")
for f in sorted(os.listdir(CFG.out_dir))[:100]:
    print("-", f)


In [1]:
# ============================================================
# CELLULE FINALE AUTONOME — Diagnostic queue basse ds_mean
# ============================================================
# Objectif :
# - Reconnexion automatique à Google Drive.
# - Lecture des fichiers *_raw.csv et *_PARTIAL_raw.csv.
# - Détection automatique de la colonne ds disponible.
# - Calcul des quantiles q05/q25/q50/q75/q95 par Z.
# - Test : ds_mean est-il tiré vers le bas par une queue basse ?
# - Export CSV/JSON/PNG + ZIP vers Drive.
# ============================================================

import os, glob, json, zipfile, shutil, warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=RuntimeWarning)

# ------------------------------------------------------------
# 0. Reconnexion Drive autonome
# ------------------------------------------------------------

IN_COLAB = False
try:
    import google.colab
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    if not os.path.exists("/content/drive/MyDrive"):
        print("[drive] Mounting Google Drive...")
        drive.mount("/content/drive", force_remount=True)
    else:
        print("[drive] Google Drive already mounted.")

# ------------------------------------------------------------
# 1. Dossiers source / sortie
# ------------------------------------------------------------

DEFAULT_DRIVE_OUTDIR = "/content/drive/MyDrive/DFGQ_Test1K/cell4_test1K_kigami_outputs"
DEFAULT_LOCAL_OUTDIR = "/content/cell4_test1K_kigami_outputs"

try:
    DRIVE_OUTDIR
except NameError:
    DRIVE_OUTDIR = DEFAULT_DRIVE_OUTDIR

try:
    OUTDIR
except NameError:
    OUTDIR = DEFAULT_LOCAL_OUTDIR

drive_exists = os.path.exists(DRIVE_OUTDIR)
local_exists = os.path.exists(OUTDIR)

if drive_exists:
    SOURCE_DIR = Path(DRIVE_OUTDIR)
elif local_exists:
    SOURCE_DIR = Path(OUTDIR)
else:
    raise RuntimeError(
        "Aucun dossier de sortie trouvé. Vérifie DRIVE_OUTDIR ou OUTDIR."
    )

DIAG_DIR = SOURCE_DIR / "FINAL_LOW_TAIL_DIAGNOSTIC"
DIAG_DIR.mkdir(parents=True, exist_ok=True)

print("SOURCE_DIR:", SOURCE_DIR)
print("DIAG_DIR:", DIAG_DIR)

# ------------------------------------------------------------
# 2. Découverte des fichiers raw
# ------------------------------------------------------------

raw_files = sorted(SOURCE_DIR.glob("*_raw.csv"))
partial_files = sorted(SOURCE_DIR.glob("*_PARTIAL_raw.csv"))

# Si un raw complet existe, il prime sur le partial du même tag.
raw_by_tag = {}

def tag_from_raw_path(p):
    name = p.name
    if name.endswith("_PARTIAL_raw.csv"):
        return name.replace("_PARTIAL_raw.csv", "")
    if name.endswith("_raw.csv"):
        return name.replace("_raw.csv", "")
    return name.replace(".csv", "")

for p in partial_files:
    raw_by_tag[tag_from_raw_path(p)] = p

for p in raw_files:
    raw_by_tag[tag_from_raw_path(p)] = p

if not raw_by_tag:
    raise RuntimeError(
        f"Aucun fichier raw trouvé dans {SOURCE_DIR}. "
        "Il faut au moins un fichier *_raw.csv ou *_PARTIAL_raw.csv."
    )

print("Raw/partial files found:", len(raw_by_tag))
for tag, p in list(raw_by_tag.items())[:20]:
    print("-", tag, "->", p.name)

# ------------------------------------------------------------
# 3. Détection automatique de la colonne ds
# ------------------------------------------------------------

PREFERRED_DS_COLUMNS = [
    # noms probables pour une valeur ds par réalisation
    "ds_eff",
    "ds_value",
    "ds",
    "ds_local",
    "ds_window",
    "ds_fit",
    "ds_mean",
    "ds_eff_mean",
    "spectral_dimension",
    "d_s",
]

def detect_ds_column(df):
    cols = list(df.columns)

    for c in PREFERRED_DS_COLUMNS:
        if c in cols and pd.api.types.is_numeric_dtype(df[c]):
            return c

    # fallback : chercher colonne numérique contenant "ds"
    candidates = []
    for c in cols:
        cl = c.lower()
        if "ds" in cl or "spectral" in cl:
            if pd.api.types.is_numeric_dtype(df[c]):
                candidates.append(c)

    if candidates:
        return candidates[0]

    return None

# ------------------------------------------------------------
# 4. Diagnostic par variante
# ------------------------------------------------------------

all_quantiles = []
all_summary = []
all_bad_rows = []

LOW_TAIL_Q50_MIN = 1.95
LOW_TAIL_Q05_MAX = 1.85
LOW_TAIL_GAP_MIN = 0.15
MEAN_BELOW_MEDIAN_MIN = 0.08

for tag, path in raw_by_tag.items():
    try:
        raw = pd.read_csv(path)
    except Exception as exc:
        all_summary.append({
            "tag": tag,
            "path": str(path),
            "status": "read_error",
            "error": str(exc),
        })
        continue

    if "Z" not in raw.columns:
        all_summary.append({
            "tag": tag,
            "path": str(path),
            "status": "missing_Z",
            "columns": list(raw.columns),
        })
        continue

    ds_col = detect_ds_column(raw)

    if ds_col is None:
        all_summary.append({
            "tag": tag,
            "path": str(path),
            "status": "missing_ds_column",
            "columns": list(raw.columns),
        })
        continue

    d = raw[["Z", ds_col]].copy()
    d["Z"] = pd.to_numeric(d["Z"], errors="coerce")
    d[ds_col] = pd.to_numeric(d[ds_col], errors="coerce")
    d = d.replace([np.inf, -np.inf], np.nan).dropna()

    if len(d) == 0:
        all_summary.append({
            "tag": tag,
            "path": str(path),
            "status": "empty_after_cleaning",
            "ds_col": ds_col,
        })
        continue

    # Quantiles par Z
    q = (
        d.groupby("Z")[ds_col]
        .quantile([0.05, 0.25, 0.50, 0.75, 0.95])
        .unstack()
        .reset_index()
        .rename(columns={
            0.05: "q05",
            0.25: "q25",
            0.50: "q50",
            0.75: "q75",
            0.95: "q95",
        })
    )

    stats = d.groupby("Z")[ds_col].agg(
        n="count",
        mean="mean",
        std="std",
        min="min",
        max="max"
    ).reset_index()

    q = q.merge(stats, on="Z", how="left")
    q["tag"] = tag
    q["source_file"] = path.name
    q["ds_col"] = ds_col

    q["q50_minus_q05"] = q["q50"] - q["q05"]
    q["q25_minus_q05"] = q["q25"] - q["q05"]
    q["mean_minus_q50"] = q["mean"] - q["q50"]
    q["q50_minus_mean"] = q["q50"] - q["mean"]

    # Critère local : médiane proche de 2 mais queue basse marquée
    q["low_tail_flag"] = (
        (q["q50"] >= LOW_TAIL_Q50_MIN) &
        (q["q05"] <= LOW_TAIL_Q05_MAX) &
        (q["q50_minus_q05"] >= LOW_TAIL_GAP_MIN)
    )

    # Critère moyen : moyenne significativement sous la médiane
    q["mean_pulled_down_flag"] = (
        q["q50_minus_mean"] >= MEAN_BELOW_MEDIAN_MIN
    )

    q["combined_low_tail_flag"] = q["low_tail_flag"] | q["mean_pulled_down_flag"]

    all_quantiles.append(q)

    # Résumé par variante
    uv = q.sort_values("Z").head(3)
    ir = q.sort_values("Z").tail(3)

    summary = {
        "tag": tag,
        "path": str(path),
        "status": "ok",
        "ds_col": ds_col,
        "n_rows": int(len(d)),
        "n_Z": int(q["Z"].nunique()),
        "Z_min": float(q["Z"].min()),
        "Z_max": float(q["Z"].max()),
        "UV_mean_first3": float(uv["mean"].mean()),
        "UV_q05_first3": float(uv["q05"].mean()),
        "UV_q50_first3": float(uv["q50"].mean()),
        "UV_q95_first3": float(uv["q95"].mean()),
        "UV_q50_minus_q05_first3": float(uv["q50_minus_q05"].mean()),
        "UV_q50_minus_mean_first3": float(uv["q50_minus_mean"].mean()),
        "IR_mean_last3": float(ir["mean"].mean()),
        "IR_q50_last3": float(ir["q50"].mean()),
        "fraction_Z_low_tail_flag": float(q["low_tail_flag"].mean()),
        "fraction_Z_mean_pulled_down_flag": float(q["mean_pulled_down_flag"].mean()),
        "fraction_Z_combined_low_tail_flag": float(q["combined_low_tail_flag"].mean()),
        "diagnosis": None,
    }

    if summary["UV_q50_first3"] >= 1.95 and summary["UV_mean_first3"] < 1.95:
        summary["diagnosis"] = "UV median near 2 but mean below 2: likely low-tail / estimator bias."
    elif summary["fraction_Z_combined_low_tail_flag"] > 0.25:
        summary["diagnosis"] = "Low-tail signature detected on a significant fraction of Z values."
    else:
        summary["diagnosis"] = "No strong low-tail signature by current thresholds."

    all_summary.append(summary)

    bad = q[q["combined_low_tail_flag"]].copy()
    if len(bad):
        all_bad_rows.append(bad)

# ------------------------------------------------------------
# 5. Exports CSV / JSON
# ------------------------------------------------------------

quantiles_df = pd.concat(all_quantiles, ignore_index=True) if all_quantiles else pd.DataFrame()
summary_df = pd.DataFrame(all_summary)
bad_df = pd.concat(all_bad_rows, ignore_index=True) if all_bad_rows else pd.DataFrame()

quantiles_path = DIAG_DIR / "LOW_TAIL_quantiles_by_Z.csv"
summary_path = DIAG_DIR / "LOW_TAIL_summary_by_variant.csv"
bad_path = DIAG_DIR / "LOW_TAIL_flagged_Z_values.csv"

quantiles_df.to_csv(quantiles_path, index=False)
summary_df.to_csv(summary_path, index=False)
bad_df.to_csv(bad_path, index=False)

global_summary = {
    "generated_at": datetime.utcnow().isoformat() + "Z",
    "source_dir": str(SOURCE_DIR),
    "diag_dir": str(DIAG_DIR),
    "n_variants_analyzed": int((summary_df["status"] == "ok").sum()) if "status" in summary_df else 0,
    "n_variants_total_seen": int(len(summary_df)),
    "thresholds": {
        "LOW_TAIL_Q50_MIN": LOW_TAIL_Q50_MIN,
        "LOW_TAIL_Q05_MAX": LOW_TAIL_Q05_MAX,
        "LOW_TAIL_GAP_MIN": LOW_TAIL_GAP_MIN,
        "MEAN_BELOW_MEDIAN_MIN": MEAN_BELOW_MEDIAN_MIN,
    },
    "interpretation": {
        "low_tail_flag": "q50 near 2 while q05 is significantly below 2 and q50-q05 is large.",
        "mean_pulled_down_flag": "q50 - mean is large, indicating the simple mean is below the typical median value.",
        "main_question": "If ds_mean << ds_median while q50 remains near 2, the low UV value of ds_mean is likely estimator/tail bias rather than a physical dimensional shift.",
    }
}

global_json_path = DIAG_DIR / "LOW_TAIL_global_summary.json"
with open(global_json_path, "w", encoding="utf-8") as f:
    json.dump(global_summary, f, indent=2, ensure_ascii=False)

print("CSV exported:")
print("-", quantiles_path)
print("-", summary_path)
print("-", bad_path)
print("-", global_json_path)

# ------------------------------------------------------------
# 6. Figures : bandes quantiles par variante
# ------------------------------------------------------------

def safe_filename(s):
    return "".join(ch if ch.isalnum() or ch in "-_." else "_" for ch in str(s))

if len(quantiles_df):
    for tag in sorted(quantiles_df["tag"].unique()):
        d = quantiles_df[quantiles_df["tag"] == tag].sort_values("Z")
        if len(d) < 3:
            continue

        fig = plt.figure(figsize=(8, 5))
        plt.fill_between(d["Z"], d["q05"], d["q95"], alpha=0.20, label="q05–q95")
        plt.fill_between(d["Z"], d["q25"], d["q75"], alpha=0.25, label="q25–q75")
        plt.plot(d["Z"], d["q50"], marker="o", linewidth=2, label="q50 median")
        plt.plot(d["Z"], d["mean"], marker="x", linewidth=1.5, label="mean")
        plt.axhline(2.0, linestyle="--", linewidth=1, label="d_s = 2")

        plt.xlabel("Z")
        plt.ylabel("spectral dimension estimate")
        plt.title(f"Low-tail diagnostic — {tag}")
        plt.legend(fontsize=8)
        plt.tight_layout()

        fig_path = DIAG_DIR / f"LOW_TAIL_quantile_band_{safe_filename(tag)}.png"
        plt.savefig(fig_path, dpi=160)
        plt.close(fig)

# Figure globale : UV mean vs UV median
if len(summary_df) and "UV_mean_first3" in summary_df.columns:
    ok = summary_df[summary_df["status"] == "ok"].copy()
    ok = ok.sort_values("tag")

    if len(ok):
        fig = plt.figure(figsize=(10, 5))
        x = np.arange(len(ok))
        plt.plot(x, ok["UV_mean_first3"], marker="o", linestyle="", label="UV mean first3")
        plt.plot(x, ok["UV_q50_first3"], marker="s", linestyle="", label="UV q50 first3")
        plt.axhline(2.0, linestyle="--", linewidth=1, label="d_s = 2")
        plt.xticks(x, ok["tag"], rotation=75, ha="right")
        plt.ylabel("UV spectral dimension estimate")
        plt.title("UV mean vs UV median — low-tail diagnostic")
        plt.legend(fontsize=8)
        plt.tight_layout()

        fig_path = DIAG_DIR / "LOW_TAIL_UV_mean_vs_median_all_variants.png"
        plt.savefig(fig_path, dpi=170)
        plt.close(fig)

# ------------------------------------------------------------
# 7. ZIP exportable
# ------------------------------------------------------------

ZIP_PATH = SOURCE_DIR / "cell4_test1K_LOW_TAIL_DIAGNOSTIC.zip"

with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(DIAG_DIR.rglob("*")):
        if p.is_file():
            zf.write(p, arcname=str(p.relative_to(DIAG_DIR)))

print("ZIP written:", ZIP_PATH)
print("ZIP size:", os.path.getsize(ZIP_PATH), "bytes")

# ------------------------------------------------------------
# 8. Affichage synthétique
# ------------------------------------------------------------

print("\n================ LOW-TAIL DIAGNOSTIC SUMMARY ================")

display_cols = [
    "tag",
    "status",
    "ds_col",
    "n_rows",
    "n_Z",
    "UV_mean_first3",
    "UV_q05_first3",
    "UV_q50_first3",
    "UV_q95_first3",
    "UV_q50_minus_q05_first3",
    "UV_q50_minus_mean_first3",
    "fraction_Z_low_tail_flag",
    "fraction_Z_mean_pulled_down_flag",
    "diagnosis",
]

display(summary_df[[c for c in display_cols if c in summary_df.columns]])

print("\nFlagged Z values:", len(bad_df))
if len(bad_df):
    display(
        bad_df[
            [
                "tag",
                "Z",
                "n",
                "mean",
                "q05",
                "q25",
                "q50",
                "q75",
                "q95",
                "q50_minus_q05",
                "q50_minus_mean",
                "low_tail_flag",
                "mean_pulled_down_flag",
            ]
        ].sort_values(["tag", "Z"]).head(100)
    )

print("\nInterpretation guide:")
print("- If UV_q50_first3 ≈ 2 but UV_mean_first3 << 2, ds_mean is likely pulled down by a low tail.")
print("- If q05 is far below q50 and q50 remains near 2, the effect is estimator/tail bias, not a physical shift of the typical dimension.")
print("- The robust publication estimator should remain ds_median unless mean, median and quantile bands all shift together.")

[drive] Mounting Google Drive...
Mounted at /content/drive
SOURCE_DIR: /content/drive/MyDrive/DFGQ_Test1K/cell4_test1K_kigami_outputs
DIAG_DIR: /content/drive/MyDrive/DFGQ_Test1K/cell4_test1K_kigami_outputs/FINAL_LOW_TAIL_DIAGNOSTIC
Raw/partial files found: 20
- base_kigami_p24 -> base_kigami_p24_raw.csv
- comparison_phenomenological -> comparison_phenomenological_raw.csv
- layers12_torus -> layers12_torus_raw.csv
- main_kigami_local -> main_kigami_local_raw.csv
- negative_plane2d -> negative_plane2d_raw.csv
- pub_layers12_torus0_p32 -> pub_layers12_torus0_p32_raw.csv
- pub_layers12_torus0_p64 -> pub_layers12_torus0_p64_raw.csv
- pub_layers12_torus1_p32 -> pub_layers12_torus1_p32_raw.csv
- pub_layers12_torus1_p64 -> pub_layers12_torus1_p64_raw.csv
- pub_layers16_torus0_p32 -> pub_layers16_torus0_p32_raw.csv
- pub_layers16_torus0_p64 -> pub_layers16_torus0_p64_raw.csv
- pub_layers16_torus1_p32 -> pub_layers16_torus1_p32_raw.csv
- pub_layers16_torus1_p64 -> pub_layers16_torus1_p64_raw.cs

/tmp/ipykernel_2823/803722950.py:314: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "generated_at": datetime.utcnow().isoformat() + "Z",


CSV exported:
- /content/drive/MyDrive/DFGQ_Test1K/cell4_test1K_kigami_outputs/FINAL_LOW_TAIL_DIAGNOSTIC/LOW_TAIL_quantiles_by_Z.csv
- /content/drive/MyDrive/DFGQ_Test1K/cell4_test1K_kigami_outputs/FINAL_LOW_TAIL_DIAGNOSTIC/LOW_TAIL_summary_by_variant.csv
- /content/drive/MyDrive/DFGQ_Test1K/cell4_test1K_kigami_outputs/FINAL_LOW_TAIL_DIAGNOSTIC/LOW_TAIL_flagged_Z_values.csv
- /content/drive/MyDrive/DFGQ_Test1K/cell4_test1K_kigami_outputs/FINAL_LOW_TAIL_DIAGNOSTIC/LOW_TAIL_global_summary.json
ZIP written: /content/drive/MyDrive/DFGQ_Test1K/cell4_test1K_kigami_outputs/cell4_test1K_LOW_TAIL_DIAGNOSTIC.zip
ZIP size: 1685132 bytes

================ LOW-TAIL DIAGNOSTIC SUMMARY ================


,tag,status,ds_col,n_rows,n_Z,UV_mean_first3,UV_q05_first3,UV_q50_first3,UV_q95_first3,UV_q50_minus_q05_first3,UV_q50_minus_mean_first3,fraction_Z_low_tail_flag,fraction_Z_mean_pulled_down_flag,diagnosis
0,base_kigami_p24,ok,ds_eff_mean,850,17,1.817068,1.792300,1.820694,1.832327,0.028394,0.003626,0.0,0.0,No strong low-tail signature by current thresh...
1,comparison_phenomenological,ok,ds_eff_mean,850,17,1.816364,1.794348,1.818512,1.833907,0.024164,0.002148,0.0,0.0,No strong low-tail signature by current thresh...
2,layers12_torus,ok,ds_eff_mean,850,17,1.816175,1.800120,1.816007,1.829820,0.015887,-0.000168,0.0,0.0,No strong low-tail signature by current thresh...
3,main_kigami_local,ok,ds_eff_mean,850,17,1.816010,1.794652,1.816779,1.833739,0.022127,0.000768,0.0,0.0,No strong low-tail signature by current thresh...
4,negative_plane2d,ok,ds_eff_mean,850,17,1.801328,1.741587,1.797375,1.859471,0.055788,-0.003953,0.0,0.0,No strong low-tail signature by current thresh...
5,pub_layers12_torus0_p32,ok,ds_eff_mean,1050,21,1.815001,1.797467,1.814857,1.830710,0.017390,-0.000144,0.0,0.0,No strong low-tail signature by current thresh...
6,pub_layers12_torus0_p64,ok,ds_eff_mean,1050,21,1.817875,1.803761,1.818371,1.830099,0.014610,0.000497,0.0,0.0,No strong low-tail signature by current thresh...
7,pub_layers12_torus1_p32,ok,ds_eff_mean,1050,21,1.818209,1.802856,1.818861,1.832351,0.016005,0.000652,0.0,0.0,No strong low-tail signature by current thresh...
8,pub_layers12_torus1_p64,ok,ds_eff_mean,1050,21,1.815971,1.801918,1.816405,1.829105,0.014487,0.000434,0.0,0.0,No strong low-tail signature by current thresh...
9,pub_layers16_torus0_p32,ok,ds_eff_mean,1050,21,1.817475,1.801840,1.818481,1.831910,0.016641,0.001006,0.0,0.0,No strong low-tail signature by current thresh...



Flagged Z values: 0

Interpretation guide:
- If UV_q50_first3 ≈ 2 but UV_mean_first3 << 2, ds_mean is likely pulled down by a low tail.
- If q05 is far below q50 and q50 remains near 2, the effect is estimator/tail bias, not a physical shift of the typical dimension.
- The robust publication estimator should remain ds_median unless mean, median and quantile bands all shift together.


In [4]:
# ============================================================
# CELLULE AUTONOME — Diagnostic intra-fenêtre via raw.csv existants
# ============================================================
# Objectif :
# - Reconnexion Drive.
# - Lecture des *_raw.csv / *_PARTIAL_raw.csv.
# - Comparaison ds_eff_mean vs ds_eff_median vs ds_eff_std.
# - Test : la moyenne de fenêtre est-elle systématiquement sous la médiane ?
# - Aucun recalcul de graphe.
# ============================================================

import os, glob, json, zipfile, warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=RuntimeWarning)

# ------------------------------------------------------------
# 0. Reconnexion Drive
# ------------------------------------------------------------

try:
    import google.colab
    from google.colab import drive
    if not os.path.exists("/content/drive/MyDrive"):
        print("[drive] Mounting Google Drive...")
        drive.mount("/content/drive", force_remount=True)
    else:
        print("[drive] Google Drive already mounted.")
except Exception:
    print("[drive] Not running in Colab or Drive unavailable.")

# ------------------------------------------------------------
# 1. Dossiers
# ------------------------------------------------------------

DRIVE_OUTDIR = "/content/drive/MyDrive/DFGQ_Test1K/cell4_test1K_kigami_outputs"
LOCAL_OUTDIR = "/content/cell4_test1K_kigami_outputs"

if os.path.exists(DRIVE_OUTDIR):
    SOURCE_DIR = Path(DRIVE_OUTDIR)
elif os.path.exists(LOCAL_OUTDIR):
    SOURCE_DIR = Path(LOCAL_OUTDIR)
else:
    raise RuntimeError("Aucun dossier de résultats trouvé.")

DIAG_DIR = SOURCE_DIR / "FINAL_WINDOW_BIAS_DIAGNOSTIC"
DIAG_DIR.mkdir(parents=True, exist_ok=True)

print("SOURCE_DIR:", SOURCE_DIR)
print("DIAG_DIR:", DIAG_DIR)

# ------------------------------------------------------------
# 2. Découverte fichiers raw
# ------------------------------------------------------------

raw_files = sorted(SOURCE_DIR.glob("*_raw.csv"))
partial_files = sorted(SOURCE_DIR.glob("*_PARTIAL_raw.csv"))

raw_by_tag = {}

def tag_from_path(p):
    name = p.name
    if name.endswith("_PARTIAL_raw.csv"):
        return name.replace("_PARTIAL_raw.csv", "")
    if name.endswith("_raw.csv"):
        return name.replace("_raw.csv", "")
    return name.replace(".csv", "")

# partial d'abord, raw complet ensuite pour priorité au complet
for p in partial_files:
    raw_by_tag[tag_from_path(p)] = p

for p in raw_files:
    raw_by_tag[tag_from_path(p)] = p

print("Raw files detected:", len(raw_by_tag))

if not raw_by_tag:
    raise RuntimeError("Aucun *_raw.csv ou *_PARTIAL_raw.csv trouvé.")

# ------------------------------------------------------------
# 3. Analyse
# ------------------------------------------------------------

expected_cols = ["Z", "realization", "ds_eff_mean", "ds_eff_median", "ds_eff_std"]

all_rows = []
all_per_z = []
all_summary = []

for tag, path in raw_by_tag.items():
    try:
        df = pd.read_csv(path)
    except Exception as exc:
        all_summary.append({
            "tag": tag,
            "status": "read_error",
            "error": str(exc),
            "path": str(path),
        })
        continue

    missing = [c for c in expected_cols if c not in df.columns]
    if missing:
        all_summary.append({
            "tag": tag,
            "status": "missing_columns",
            "missing": ",".join(missing),
            "columns": ",".join(df.columns),
            "path": str(path),
        })
        continue

    d = df[expected_cols].copy()
    for c in ["Z", "realization", "ds_eff_mean", "ds_eff_median", "ds_eff_std"]:
        d[c] = pd.to_numeric(d[c], errors="coerce")

    d = d.replace([np.inf, -np.inf], np.nan).dropna()
    if len(d) == 0:
        all_summary.append({
            "tag": tag,
            "status": "empty_after_cleaning",
            "path": str(path),
        })
        continue

    d["tag"] = tag
    d["source_file"] = path.name
    d["median_minus_mean"] = d["ds_eff_median"] - d["ds_eff_mean"]
    d["relative_bias"] = d["median_minus_mean"] / d["ds_eff_median"].replace(0, np.nan)
    d["mean_below_median_flag"] = d["median_minus_mean"] > 0.10
    d["strong_mean_below_median_flag"] = d["median_minus_mean"] > 0.20
    d["large_window_std_flag"] = d["ds_eff_std"] > 0.35

    all_rows.append(d)

    per_z = (
        d.groupby("Z", as_index=False)
        .agg(
            n_rows=("ds_eff_mean", "count"),
            mean_ds_eff_mean=("ds_eff_mean", "mean"),
            median_ds_eff_mean=("ds_eff_mean", "median"),
            mean_ds_eff_median=("ds_eff_median", "mean"),
            median_ds_eff_median=("ds_eff_median", "median"),
            mean_ds_eff_std=("ds_eff_std", "mean"),
            median_minus_mean_avg=("median_minus_mean", "mean"),
            median_minus_mean_median=("median_minus_mean", "median"),
            relative_bias_avg=("relative_bias", "mean"),
            frac_mean_below_median=("mean_below_median_flag", "mean"),
            frac_strong_mean_below_median=("strong_mean_below_median_flag", "mean"),
            frac_large_window_std=("large_window_std_flag", "mean"),
        )
    )
    per_z["tag"] = tag
    per_z["source_file"] = path.name
    all_per_z.append(per_z)

    uv = per_z.sort_values("Z").head(3)
    ir = per_z.sort_values("Z").tail(3)

    all_summary.append({
        "tag": tag,
        "status": "ok",
        "source_file": path.name,
        "n_rows": int(len(d)),
        "n_Z": int(per_z["Z"].nunique()),
        "Z_min": float(per_z["Z"].min()),
        "Z_max": float(per_z["Z"].max()),
        "UV_mean_ds_eff_mean_first3": float(uv["mean_ds_eff_mean"].mean()),
        "UV_mean_ds_eff_median_first3": float(uv["mean_ds_eff_median"].mean()),
        "UV_median_minus_mean_first3": float(uv["median_minus_mean_avg"].mean()),
        "UV_ds_eff_std_first3": float(uv["mean_ds_eff_std"].mean()),
        "IR_mean_ds_eff_mean_last3": float(ir["mean_ds_eff_mean"].mean()),
        "IR_mean_ds_eff_median_last3": float(ir["mean_ds_eff_median"].mean()),
        "IR_median_minus_mean_last3": float(ir["median_minus_mean_avg"].mean()),
        "global_median_minus_mean_avg": float(per_z["median_minus_mean_avg"].mean()),
        "global_ds_eff_std_avg": float(per_z["mean_ds_eff_std"].mean()),
        "fraction_Z_with_strong_bias": float((per_z["median_minus_mean_avg"] > 0.20).mean()),
        "fraction_Z_with_large_std": float((per_z["mean_ds_eff_std"] > 0.35).mean()),
        "diagnosis": (
            "intra_window_bias_likely"
            if float(uv["median_minus_mean_avg"].mean()) > 0.20
            else "weak_or_no_intra_window_bias"
        ),
    })

raw_diag = pd.concat(all_rows, ignore_index=True) if all_rows else pd.DataFrame()
per_z_diag = pd.concat(all_per_z, ignore_index=True) if all_per_z else pd.DataFrame()
summary_diag = pd.DataFrame(all_summary)

# ------------------------------------------------------------
# 4. Exports
# ------------------------------------------------------------

raw_path = DIAG_DIR / "WINDOW_BIAS_raw_with_flags.csv"
per_z_path = DIAG_DIR / "WINDOW_BIAS_per_Z_summary.csv"
summary_path = DIAG_DIR / "WINDOW_BIAS_summary_by_variant.csv"
status_path = DIAG_DIR / "WINDOW_BIAS_status.json"

raw_diag.to_csv(raw_path, index=False)
per_z_diag.to_csv(per_z_path, index=False)
summary_diag.to_csv(summary_path, index=False)

status = {
    "generated_at": datetime.utcnow().isoformat() + "Z",
    "source_dir": str(SOURCE_DIR),
    "diag_dir": str(DIAG_DIR),
    "n_variants_seen": int(len(raw_by_tag)),
    "n_variants_ok": int((summary_diag["status"] == "ok").sum()) if "status" in summary_diag else 0,
    "method": "Uses existing raw.csv columns ds_eff_mean, ds_eff_median, ds_eff_std. No graph recomputation.",
    "interpretation": {
        "median_minus_mean": "Positive values mean the diffusion-window mean is below the diffusion-window median.",
        "large_ds_eff_std": "Large ds_eff_std indicates broad intra-window variation of local d_s(u).",
        "if_UV_bias": "If UV ds_eff_median is near 2 while ds_eff_mean is near 1.8, the issue is intra-window averaging, not inter-realization low-tail.",
    }
}

with open(status_path, "w", encoding="utf-8") as f:
    json.dump(status, f, indent=2, ensure_ascii=False)

# ------------------------------------------------------------
# 5. Figures
# ------------------------------------------------------------

if len(per_z_diag):
    for tag in sorted(per_z_diag["tag"].unique()):
        d = per_z_diag[per_z_diag["tag"] == tag].sort_values("Z")
        if len(d) < 3:
            continue

        fig = plt.figure(figsize=(8, 5))
        plt.plot(d["Z"], d["mean_ds_eff_mean"], marker="o", label="mean of ds_eff_mean")
        plt.plot(d["Z"], d["mean_ds_eff_median"], marker="s", label="mean of ds_eff_median")
        plt.axhline(2.0, linestyle="--", linewidth=1, label="d_s = 2")
        plt.xlabel("Z")
        plt.ylabel("effective spectral dimension")
        plt.title(f"Window estimator bias — {tag}")
        plt.legend(fontsize=8)
        plt.tight_layout()
        plt.savefig(DIAG_DIR / f"WINDOW_BIAS_mean_vs_median_{tag}.png", dpi=160)
        plt.close(fig)

        fig = plt.figure(figsize=(8, 5))
        plt.plot(d["Z"], d["median_minus_mean_avg"], marker="o", label="median - mean")
        plt.plot(d["Z"], d["mean_ds_eff_std"], marker="x", label="ds_eff_std")
        plt.axhline(0.20, linestyle="--", linewidth=1, label="bias threshold 0.20")
        plt.xlabel("Z")
        plt.ylabel("bias / std")
        plt.title(f"Intra-window spread — {tag}")
        plt.legend(fontsize=8)
        plt.tight_layout()
        plt.savefig(DIAG_DIR / f"WINDOW_BIAS_spread_{tag}.png", dpi=160)
        plt.close(fig)

    # synthèse UV
    ok = summary_diag[summary_diag["status"] == "ok"].copy()
    if len(ok):
        ok = ok.sort_values("tag")
        x = np.arange(len(ok))

        fig = plt.figure(figsize=(11, 5))
        plt.plot(x, ok["UV_mean_ds_eff_mean_first3"], marker="o", linestyle="", label="UV ds_eff_mean")
        plt.plot(x, ok["UV_mean_ds_eff_median_first3"], marker="s", linestyle="", label="UV ds_eff_median")
        plt.axhline(2.0, linestyle="--", linewidth=1, label="d_s = 2")
        plt.xticks(x, ok["tag"], rotation=75, ha="right", fontsize=8)
        plt.ylabel("UV effective d_s")
        plt.title("UV mean vs median estimator")
        plt.legend(fontsize=8)
        plt.tight_layout()
        plt.savefig(DIAG_DIR / "WINDOW_BIAS_UV_mean_vs_median_all_variants.png", dpi=180)
        plt.close(fig)

# ------------------------------------------------------------
# 6. ZIP
# ------------------------------------------------------------

ZIP_PATH = SOURCE_DIR / "cell4_test1K_WINDOW_BIAS_DIAGNOSTIC.zip"

with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(DIAG_DIR.rglob("*")):
        if p.is_file():
            zf.write(p, arcname=str(p.relative_to(DIAG_DIR)))

print("ZIP written:", ZIP_PATH)
print("ZIP size:", os.path.getsize(ZIP_PATH), "bytes")

# ------------------------------------------------------------
# 7. Affichage
# ------------------------------------------------------------

print("\n================ WINDOW BIAS DIAGNOSTIC ================")
display_cols = [
    "tag",
    "status",
    "n_rows",
    "n_Z",
    "UV_mean_ds_eff_mean_first3",
    "UV_mean_ds_eff_median_first3",
    "UV_median_minus_mean_first3",
    "UV_ds_eff_std_first3",
    "global_median_minus_mean_avg",
    "global_ds_eff_std_avg",
    "fraction_Z_with_strong_bias",
    "diagnosis",
]

display(summary_diag[[c for c in display_cols if c in summary_diag.columns]])

print("\nMain files:")
print("-", summary_path)
print("-", per_z_path)
print("-", raw_path)
print("-", ZIP_PATH)

[drive] Google Drive already mounted.
SOURCE_DIR: /content/drive/MyDrive/DFGQ_Test1K/cell4_test1K_kigami_outputs
DIAG_DIR: /content/drive/MyDrive/DFGQ_Test1K/cell4_test1K_kigami_outputs/FINAL_WINDOW_BIAS_DIAGNOSTIC
Raw files detected: 20


/tmp/ipykernel_2823/2726192700.py:210: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "generated_at": datetime.utcnow().isoformat() + "Z",


ZIP written: /content/drive/MyDrive/DFGQ_Test1K/cell4_test1K_kigami_outputs/cell4_test1K_WINDOW_BIAS_DIAGNOSTIC.zip
ZIP size: 3304153 bytes

================ WINDOW BIAS DIAGNOSTIC ================


,tag,status,n_rows,n_Z,UV_mean_ds_eff_mean_first3,UV_mean_ds_eff_median_first3,UV_median_minus_mean_first3,UV_ds_eff_std_first3,global_median_minus_mean_avg,global_ds_eff_std_avg,fraction_Z_with_strong_bias,diagnosis
0,base_kigami_p24,ok,850,17,1.817068,2.084827,0.267759,0.584729,0.236525,0.769086,0.529412,intra_window_bias_likely
1,comparison_phenomenological,ok,850,17,1.816364,2.080525,0.264161,0.584457,0.262379,0.764094,0.941176,intra_window_bias_likely
2,layers12_torus,ok,850,17,1.816175,2.080482,0.264307,0.583158,0.217054,0.805801,0.470588,intra_window_bias_likely
3,main_kigami_local,ok,850,17,1.816010,2.080075,0.264064,0.583507,0.235633,0.770105,0.529412,intra_window_bias_likely
4,negative_plane2d,ok,850,17,1.801328,2.099941,0.298614,0.604027,0.297847,0.601872,1.000000,intra_window_bias_likely
5,pub_layers12_torus0_p32,ok,1050,21,1.815001,2.074185,0.259185,0.580909,0.228714,0.777835,0.523810,intra_window_bias_likely
6,pub_layers12_torus0_p64,ok,1050,21,1.817875,2.076088,0.258214,0.581933,0.227289,0.777605,0.523810,intra_window_bias_likely
7,pub_layers12_torus1_p32,ok,1050,21,1.818209,2.081661,0.263452,0.583988,0.216178,0.803479,0.476190,intra_window_bias_likely
8,pub_layers12_torus1_p64,ok,1050,21,1.815971,2.073173,0.257202,0.580399,0.216964,0.802956,0.476190,intra_window_bias_likely
9,pub_layers16_torus0_p32,ok,1050,21,1.817475,2.076927,0.259451,0.581902,0.226247,0.783614,0.523810,intra_window_bias_likely



Main files:
- /content/drive/MyDrive/DFGQ_Test1K/cell4_test1K_kigami_outputs/FINAL_WINDOW_BIAS_DIAGNOSTIC/WINDOW_BIAS_summary_by_variant.csv
- /content/drive/MyDrive/DFGQ_Test1K/cell4_test1K_kigami_outputs/FINAL_WINDOW_BIAS_DIAGNOSTIC/WINDOW_BIAS_per_Z_summary.csv
- /content/drive/MyDrive/DFGQ_Test1K/cell4_test1K_kigami_outputs/FINAL_WINDOW_BIAS_DIAGNOSTIC/WINDOW_BIAS_raw_with_flags.csv
- /content/drive/MyDrive/DFGQ_Test1K/cell4_test1K_kigami_outputs/cell4_test1K_WINDOW_BIAS_DIAGNOSTIC.zip


# Lecture recommandée du résultat

Le résultat attendu n’est pas que la règle de résistance locale “prouve” Kigami au sens strict. Le statut correct est :

\[
\boxed{\text{Test spectral fondé sur une forme de Dirichlet effective.}}
\]

Le succès reviewer-safe minimal est :

1. le contrôle 2D reste compatible avec un profil plat ;
2. le multiplex à conductances Kigami-locales produit un \(d_s(Z)\) sigmoïdal ;
3. le modèle sigmoïde/logistique/Hill est préféré à constant, linéaire et puissance pure ;
4. les variantes de robustesse conservent \(Z_{\rm th}\), \(\Delta Z\), \(d_{\rm IR}\) dans une plage stable ;
5. le comparateur Hill montre que la “croissance de puissance saturante” n’est pas une alternative ignorée mais une autre écriture de la logistique en \(Z\).

Phrase intégrable :

> The Kigami-weighted extension of Test 1 replaces the phenomenological vertical weights by local resistance-conductance proxies \(c_{ij}^{(n)}\sim1/R_{ij}^{(n)}\), while preserving the same heat-trace extraction of \(d_s^{eff}(Z)\). The resulting protocol does not claim a full analytical derivation of the resistance form, but it upgrades the spectral crossover into a Dirichlet-energy-based numerical test. A Hill-type saturating power law is included as an explicit bounded-power competitor; in the logarithmic scale \(Z\), it is equivalent to a logistic crossover.
